# CUDA Monte Carlo GBM stock-price simulator (CPU baseline, custom kernel, benchmarks)

Self-contained Kaggle workflow for a GPU-computing portfolio project.

**What it does**

1. Environment check: GPU model, compute capability, `nvcc`, driver and runtime versions.
2. Compiles `src/gbm_montecarlo.cu` (CPU baseline + a custom `__global__` CUDA kernel),
   then runs `smoke` and the statistical correctness suite.
3. Benchmarks CPU vs GPU across 10k / 100k / 1M paths and CUDA block sizes
   64 / 128 / 256 / 512, writing raw CSV.
4. Plots the results, then profiles: it uses NVIDIA Nsight Systems / Compute if this image has
   them and the container permits them, and otherwise falls back to whatever works here - a
   measured CUDA-event run of the real kernel, static kernel statistics read out of the built
   cubin (`cuobjdump`, `ptxas`), the occupancy derived from them, and a sampled `nvidia-smi`
   telemetry trace. Every finding is labelled measured / static / estimated in
   `results/profiling.md`, together with what is *not* available and why.
5. Verifies the whole result set (every artifact present, valid CSV, no failed check) and only
   then packages it into `gbm_montecarlo_results.zip` and offers the files for download.

**Model** (only the terminal price is stored; one CUDA thread per independent path):

    S(t+dt) = S(t) * exp((mu - 0.5*sigma^2)*dt + sigma*sqrt(dt)*Z),   Z ~ N(0,1),  dt = T/steps

**Requirements:** a CUDA-capable GPU (Kaggle GPU session: T4 = `sm_75`, P100 = `sm_60`) and the
CUDA toolkit (`nvcc`). No extra `pip install` is needed on a Kaggle GPU image.

Everything is written under the current working directory: `src/`, `build/`, `results/`.

Run the cells in order (**Run All**). Cell 5 refuses to download anything unless cells 1-4
produced a complete, passing result set.

In [ ]:
# ============================== 1. Environment ==============================
import json, os, pathlib, re, shutil, subprocess, sys, time

RUN_START = time.time()  # cell 5 compares artifact timestamps against this

ROOT = pathlib.Path.cwd()
for d in ("src", "results", "build"):
    (ROOT / d).mkdir(parents=True, exist_ok=True)
print("working dir :", ROOT)
print("python      :", sys.version.split()[0])

ENV_LOG = []


def run(cmd, quiet=False, timeout=1800):
    '''Run a command, echo it, return CompletedProcess (stdout+stderr captured).'''
    printable = cmd if isinstance(cmd, str) else " ".join(cmd)
    if not quiet:
        print("$", printable, flush=True)
    p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True,
                       timeout=timeout)
    if not quiet:
        if p.stdout.strip():
            print(p.stdout.rstrip())
        if p.stderr.strip():
            print(p.stderr.rstrip()[:3000])
    return p


def banner(t):
    print("\n" + "=" * 74 + "\n" + t + "\n" + "=" * 74)


# ---- GPU -------------------------------------------------------------------
smi = run(["nvidia-smi"], quiet=False)
ENV_LOG.append("$ nvidia-smi\n" + (smi.stdout or "") + (smi.stderr or ""))

GPU_NAME, ARCH = None, None
q = run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total,driver_version",
         "--format=csv,noheader"], quiet=False)
ENV_LOG.append("$ nvidia-smi --query-gpu=...\n" + (q.stdout or "") + (q.stderr or ""))
try:
    first = [r for r in q.stdout.strip().splitlines() if r.strip()][0]
    parts = [x.strip() for x in first.split(",")]
    GPU_NAME = parts[0]
    if len(parts) > 1 and "." in parts[1]:
        major, minor = parts[1].split(".")
        ARCH = "sm_%s%s" % (major, minor)
except Exception as exc:  # pragma: no cover - driver dependent
    print("compute_cap query failed:", exc)

if ARCH is None:
    try:
        import torch
        cap = torch.cuda.get_device_capability(0)
        ARCH = "sm_%d%d" % (cap[0], cap[1])
        GPU_NAME = torch.cuda.get_device_name(0)
        print("arch via torch:", ARCH)
    except Exception as exc:
        print("torch fallback failed:", exc)

# ---- CPU -------------------------------------------------------------------
cpu_model, n_cpu = "", str(os.cpu_count())
try:
    for line in open("/proc/cpuinfo"):
        if line.lower().startswith("model name"):
            cpu_model = line.split(":", 1)[1].strip()
            break
except OSError:
    pass
n_cpu = os.cpu_count()
print("\nGPU        :", GPU_NAME, "| arch:", ARCH)
print("CPU        :", cpu_model, "| logical cores:", n_cpu)

# ---- CUDA toolkit ----------------------------------------------------------
NVCC = shutil.which("nvcc")
if NVCC is None and pathlib.Path("/usr/local/cuda/bin/nvcc").exists():
    NVCC = "/usr/local/cuda/bin/nvcc"
EXTRA_FLAGS = []

if NVCC is None:
    print("\nnvcc not found in the image; trying the pip CUDA toolchain fallback ...")
    run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cuda-nvcc-cu12",
         "nvidia-cuda-runtime-cu12", "nvidia-curand-cu12"])
    import site
    search = list(site.getsitepackages()) + [site.getusersitepackages()] + [ROOT]
    for sp in search:
        cand = pathlib.Path(sp) / "nvidia" / "cuda_nvcc" / "bin" / "nvcc"
        if cand.exists():
            NVCC = str(cand)
            break
    for sp in search:
        base = pathlib.Path(sp) / "nvidia"
        if not base.exists():
            continue
        for inc in ("cuda_runtime/include", "curand/include", "cuda_nvcc/include"):
            if (base / inc).exists():
                EXTRA_FLAGS += ["-I", str(base / inc)]
        for lib in ("cuda_runtime/lib", "curand/lib"):
            if (base / lib).exists():
                EXTRA_FLAGS += ["-L", str(base / lib)]

assert NVCC is not None, "no nvcc available - use a Kaggle GPU image (or install the CUDA toolkit)"
ver = run([NVCC, "--version"])
ENV_LOG.append("$ nvcc --version\n" + (ver.stdout or "") + (ver.stderr or ""))
assert ARCH is not None, "could not determine the GPU compute capability"

print("\nnvcc       :", NVCC)
print("arch flag  : -arch=%s" % ARCH)
print("extra flags:", EXTRA_FLAGS or "(none)")

ENV_LOG.append("GPU: %s\narch: %s\nCPU: %s (%s logical cores)\n" % (GPU_NAME, ARCH, cpu_model, n_cpu))
(pathlib.Path("results") / "environment.txt").write_text(
    "=== nvidia-smi ===\n" + (smi.stdout or smi.stderr) + "\n" + "\n".join(ENV_LOG))
print("\nwrote results/environment.txt")

In [ ]:
# ================== 2. Source, compile, correctness =========================
import pathlib, subprocess, time

SOURCE = r'''
// =============================================================================
//  Monte Carlo Geometric Brownian Motion (GBM) terminal-price simulator
//  ---------------------------------------------------------------------------
//  One self-contained translation unit that provides:
//    1. A single-threaded CPU baseline  (std::mt19937_64 + Box-Muller)
//    2. A custom CUDA kernel           (curandStatePhilox4_32_10_t, __global__)
//    3. A numerical-correctness harness (analytical log-normal moments, z-scores)
//    4. A benchmark harness            (CUDA events, medians, CSV output)
//
//  Model (real-world drift mu; the Euler update is exact in distribution for
//  the terminal value, so any steps >= 1 gives the same S(T) law):
//      S(t+dt) = S(t) * exp( (mu - 0.5*sigma^2)*dt + sigma*sqrt(dt)*Z ),
//      Z ~ N(0,1),  dt = T / steps
//  Only the terminal price S(T) is stored: one thread per independent path.
//
//  ---------------------------------------------------------------------------
//  Build (GPU, requires nvcc + CUDA toolkit):
//      nvcc -O3 -std=c++17 -arch=sm_75 -lineinfo \
//           -o gbm src/gbm_montecarlo.cu -lcurand
//
//  Build (CPU only, no CUDA toolkit needed -- used to sanity-check this file):
//      g++ -x c++ -O3 -std=c++11 -DGBM_CPU_ONLY -o gbm_cpu src/gbm_montecarlo.cu
//
//  ---------------------------------------------------------------------------
//  Usage:
//      ./gbm info                          # device + runtime information
//      ./gbm smoke                         # fast end-to-end sanity run
//      ./gbm verify   [--paths N] [--seeds K] [--steps S] [--out DIR]
//      ./gbm bench    [--sizes a,b,c] [--blocks a,b,c] [--steps S]
//                     [--reps R] [--cpu-reps R] [--out DIR]
//      ./gbm rng      [--paths N] [--block B] [--steps a,b,c] [--reps R] [--out DIR]
//
//  Every command writes machine-readable CSV into --out (default: results/).
//  No measurement in this file is invented: all numbers are timed at runtime.
// =============================================================================

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <random>
#include <string>
#include <vector>

#if defined(_WIN32)
#  include <direct.h>
#  define GBM_MKDIR(p) _mkdir(p)
#else
#  include <sys/stat.h>
#  include <sys/types.h>
#  define GBM_MKDIR(p) mkdir((p), 0755)
#endif

#ifdef GBM_CPU_ONLY
#  define GBM_HAS_CUDA 0
#else
#  define GBM_HAS_CUDA 1
#  include <cuda_runtime.h>
#  include <curand_kernel.h>
#endif

#define GBM_PI 3.14159265358979323846

// -----------------------------------------------------------------------------
//  Small utilities
// -----------------------------------------------------------------------------

static std::string g_out_dir = "results";

// Portable recursive mkdir (no <filesystem>, so this also builds with older
// toolchains). Existing directories are ignored; failures surface later as an
// explicit "cannot write <file>" message from the writer that needed the dir.
static void ensure_dir(const std::string& path) {
  if (path.empty()) return;
  std::string cur;
  for (size_t i = 0; i < path.size(); ++i) {
    const char c = path[i];
    if (c == '/' || c == '\\') {
      if (!cur.empty()) GBM_MKDIR(cur.c_str());
    }
    cur += c;
  }
  if (!cur.empty()) GBM_MKDIR(cur.c_str());
}

static std::string path_in_out(const std::string& fname) { return g_out_dir + "/" + fname; }

static double now_ms() {
  using clock = std::chrono::steady_clock;
  static const clock::time_point t0 = clock::now();
  return std::chrono::duration<double, std::milli>(clock::now() - t0).count();
}

// Median of a copy of the samples. Even counts -> mean of the two middle values.
static double median(std::vector<double> v) {
  if (v.empty()) return 0.0;
  std::sort(v.begin(), v.end());
  size_t n = v.size();
  return (n % 2 == 1) ? v[n / 2] : 0.5 * (v[n / 2 - 1] + v[n / 2]);
}

static double mean_of(const std::vector<double>& v) {
  if (v.empty()) return 0.0;
  double s = 0.0;
  for (size_t i = 0; i < v.size(); ++i) s += v[i];
  return s / (double)v.size();
}

static std::string get_opt(int argc, char** argv, const char* key, const std::string& def) {
  for (int i = 1; i + 1 < argc; ++i)
    if (std::strcmp(argv[i], key) == 0) return std::string(argv[i + 1]);
  return def;
}
static long long get_opt_ll(int argc, char** argv, const char* key, long long def) {
  std::string s = get_opt(argc, argv, key, "");
  return s.empty() ? def : std::atoll(s.c_str());
}
static std::vector<long long> parse_int_list(const std::string& s) {
  std::vector<long long> out;
  size_t i = 0;
  while (i < s.size()) {
    size_t j = s.find(',', i);
    if (j == std::string::npos) j = s.size();
    std::string tok = s.substr(i, j - i);
    if (!tok.empty()) out.push_back(std::atoll(tok.c_str()));
    i = j + 1;
  }
  return out;
}
static std::string join_csv(const std::vector<long long>& v) {
  std::string s;
  for (size_t i = 0; i < v.size(); ++i) {
    if (i) s += ",";
    char buf[32];
    std::snprintf(buf, sizeof(buf), "%lld", v[i]);
    s += buf;
  }
  return s;
}

// -----------------------------------------------------------------------------
//  Simulation parameters and analytical (log-normal) reference moments
// -----------------------------------------------------------------------------

struct SimParams {
  double S0;     // spot
  double mu;     // annualised drift
  double sigma;  // annualised volatility
  double T;      // horizon in years
  int steps;     // time steps per path (dt = T / steps)
};

// Our process has log-drift (mu - 0.5*sigma^2), so
//   E[S(T)^k] = S0^k * exp(k*mu*T + 0.5*k*(k-1)*sigma^2*T).
// k = 1 gives the textbook E[S(T)] = S0*exp(mu*T) used as the primary check.
static double raw_moment(const SimParams& p, int k) {
  return std::pow(p.S0, (double)k) *
         std::exp((double)k * p.mu * p.T +
                  0.5 * (double)k * (double)(k - 1) * p.sigma * p.sigma * p.T);
}
static double analytic_mean(const SimParams& p) { return raw_moment(p, 1); }
static double analytic_var(const SimParams& p) {
  double m1 = raw_moment(p, 1), m2 = raw_moment(p, 2);
  return m2 - m1 * m1;
}
// Central 4th moment, built from raw moments (numerically stable enough here).
static double analytic_central_m4(const SimParams& p) {
  double m1 = raw_moment(p, 1), m2 = raw_moment(p, 2), m3 = raw_moment(p, 3), m4 = raw_moment(p, 4);
  return m4 - 4.0 * m1 * m3 + 6.0 * m1 * m1 * m2 - 3.0 * m1 * m1 * m1 * m1;
}
// Standard error of the sample mean and of the unbiased sample variance.
static double se_mean(const SimParams& p, long long n) { return std::sqrt(analytic_var(p) / (double)n); }
static double se_var(const SimParams& p, long long n) {
  double v = analytic_var(p);
  double c = analytic_central_m4(p) - v * v;
  return (c > 0.0) ? std::sqrt(c / (double)n) : 0.0;
}

// -----------------------------------------------------------------------------
//  Empirical statistics of a set of terminal prices
// -----------------------------------------------------------------------------

struct Stats {
  long long n;
  long long n_nonfinite;
  long long n_nonpositive;
  double mean;
  double var;  // unbiased (n-1)
  double sd;
  double min;
  double max;
};

static Stats compute_stats(const std::vector<float>& v) {
  Stats s;
  s.n = (long long)v.size();
  s.n_nonfinite = 0;
  s.n_nonpositive = 0;
  s.mean = 0.0;
  s.var = 0.0;
  s.sd = 0.0;
  s.min = 0.0;
  s.max = 0.0;
  if (v.empty()) return s;
  // Accumulate in double regardless of storage type (float prices -> exact sums).
  double sum = 0.0, sumsq = 0.0, mn = 0.0, mx = 0.0;
  bool first = true;
  for (size_t i = 0; i < v.size(); ++i) {
    double x = (double)v[i];
    if (!std::isfinite(x)) {
      s.n_nonfinite++;
      continue;
    }
    if (x <= 0.0) s.n_nonpositive++;
    sum += x;
    sumsq += x * x;
    if (first) {
      mn = mx = x;
      first = false;
    } else {
      mn = std::min(mn, x);
      mx = std::max(mx, x);
    }
  }
  long long m = s.n - s.n_nonfinite;
  s.min = first ? 0.0 : mn;
  s.max = first ? 0.0 : mx;
  if (m > 0) {
    s.mean = sum / (double)m;
    if (m > 1) {
      double c = sumsq - (double)m * s.mean * s.mean;
      if (c < 0.0) c = 0.0;
      s.var = c / (double)(m - 1);
    }
    s.sd = std::sqrt(s.var);
  }
  return s;
}

// -----------------------------------------------------------------------------
//  CPU baseline
//  std::mt19937_64 (Mersenne Twister, 2^19937-1 period) + Box-Muller.
//  All arithmetic in double; the GPU path uses float, so the two are compared
//  statistically (distributional agreement), never element-by-element.
// -----------------------------------------------------------------------------

// Box-Muller generator over std::mt19937_64. Both sin/cos outputs are used, so
// each (log, sqrt, sin, cos) block yields two normal deviates (one is cached).
struct BoxMuller {
  std::mt19937_64 gen;
  bool has_spare;
  double spare;
  explicit BoxMuller(uint64_t seed) : gen(seed), has_spare(false), spare(0.0) {}
  inline double uniform() {
    // 53 random bits -> uniform double in [0,1).
    return (double)(gen() >> 11) * (1.0 / 9007199254740992.0);
  }
  inline double normal() {
    if (has_spare) {
      has_spare = false;
      return spare;
    }
    double u1;
    do {
      u1 = uniform();
    } while (u1 <= 1e-300);  // guard log(0)
    const double u2 = uniform();
    const double r = std::sqrt(-2.0 * std::log(u1));
    const double th = 2.0 * GBM_PI * u2;
    spare = r * std::sin(th);
    has_spare = true;
    return r * std::cos(th);
  }
};

static double cpu_simulate(const SimParams& p, long long n_paths, uint64_t seed,
                           std::vector<float>& out) {
  out.assign((size_t)n_paths, 0.0f);
  const double dt = p.T / (double)p.steps;
  const double drift = (p.mu - 0.5 * p.sigma * p.sigma) * dt;
  const double vol = p.sigma * std::sqrt(dt);

  BoxMuller rng(seed);
  const double t0 = now_ms();
  for (long long i = 0; i < n_paths; ++i) {
    double s = p.S0;
    for (int t = 0; t < p.steps; ++t) s *= std::exp(drift + vol * rng.normal());
    out[(size_t)i] = (float)s;
  }
  return now_ms() - t0;
}

// -----------------------------------------------------------------------------
//  Custom CUDA kernel
//  One thread == one independent Monte Carlo path. Threads are stored in a
//  per-thread cuRAND Philox state (counter-based, high quality, cheap to seed
//  with a distinct subsequence per path). The `i >= n_paths` guard makes any
//  path count legal, including counts that are not a multiple of blockDim.
// -----------------------------------------------------------------------------
#if GBM_HAS_CUDA

__global__ void gbm_terminal_price_kernel(float* __restrict__ out,
                                         const long long n_paths,
                                         const int steps,
                                         const float S0,
                                         const float drift,  // (mu - 0.5 sigma^2) * dt
                                         const float vol,    // sigma * sqrt(dt)
                                         const unsigned long long seed) {
  const long long i = (long long)blockIdx.x * (long long)blockDim.x + (long long)threadIdx.x;
  if (i >= n_paths) return;  // handles non-multiples of the block size

  curandStatePhilox4_32_10_t st;
  curand_init(seed, (unsigned long long)i, 0ULL, &st);

  float s = S0;
  const int pairs = steps >> 1;
#pragma unroll 4
  for (int k = 0; k < pairs; ++k) {
    // Two N(0,1) deviates per Philox counter step: halves RNG overhead.
    const float2 z = curand_normal2(&st);
    s *= expf(drift + vol * z.x);
    s *= expf(drift + vol * z.y);
  }
  if (steps & 1) {
    const float z = curand_normal(&st);
    s *= expf(drift + vol * z);
  }
  out[i] = s;
}

static bool cuda_check(cudaError_t rc, const char* what, std::string* err) {
  if (rc != cudaSuccess) {
    if (err) *err = std::string(what) + ": " + cudaGetErrorString(rc);
    return false;
  }
  return true;
}

enum GpuMeasure { GPU_KERNEL_ONLY, GPU_END_TO_END };

// Runs the kernel once. GPU_KERNEL_ONLY times only the kernel with CUDA events;
// GPU_END_TO_END times cudaMalloc + launch + D2H copy + cudaFree.
static bool gpu_run_once(const SimParams& p, long long n_paths, int block, uint64_t seed,
                         GpuMeasure mode, std::vector<float>& out, double* ms, std::string* err) {
  out.assign((size_t)n_paths, 0.0f);
  *ms = -1.0;
  if (n_paths <= 0) {
    *ms = 0.0;
    return true;
  }
  const float dt = (float)(p.T / (double)p.steps);
  const float drift = (float)((p.mu - 0.5 * p.sigma * p.sigma) * (double)dt);
  const float vol = (float)(p.sigma * std::sqrt((double)dt));
  const size_t bytes = (size_t)n_paths * sizeof(float);
  const unsigned grid = (unsigned)((n_paths + (long long)block - 1) / (long long)block);

  cudaEvent_t e0 = NULL, e1 = NULL;
  if (!cuda_check(cudaEventCreate(&e0), "cudaEventCreate", err) ||
      !cuda_check(cudaEventCreate(&e1), "cudaEventCreate", err))
    return false;

  float* d_out = NULL;
  const double t_e2e = (mode == GPU_END_TO_END) ? now_ms() : 0.0;
  if (!cuda_check(cudaMalloc(&d_out, bytes), "cudaMalloc", err)) return false;

  if (mode == GPU_KERNEL_ONLY) cuda_check(cudaEventRecord(e0, 0), "cudaEventRecord", err);
  gbm_terminal_price_kernel<<<grid, block>>>(d_out, n_paths, p.steps, (float)p.S0, drift, vol,
                                            (unsigned long long)seed);
  if (mode == GPU_KERNEL_ONLY) {
    cuda_check(cudaEventRecord(e1, 0), "cudaEventRecord", err);
    cuda_check(cudaEventSynchronize(e1), "cudaEventSynchronize", err);
    float el = 0.0f;
    cuda_check(cudaEventElapsedTime(&el, e0, e1), "cudaEventElapsedTime", err);
    *ms = (double)el;
  }

  // Always validate the launch (also catches async kernel faults before the copy).
  if (!cuda_check(cudaPeekAtLastError(), "kernel launch", err)) {
    cudaFree(d_out);
    cudaEventDestroy(e0);
    cudaEventDestroy(e1);
    return false;
  }
  if (!cuda_check(cudaMemcpy(&out[0], d_out, bytes, cudaMemcpyDeviceToHost), "cudaMemcpy D2H",
                  err)) {
    cudaFree(d_out);
    cudaEventDestroy(e0);
    cudaEventDestroy(e1);
    return false;
  }

  if (mode == GPU_END_TO_END) *ms = now_ms() - t_e2e;

  cudaFree(d_out);
  cudaEventDestroy(e0);
  cudaEventDestroy(e1);
  return err == NULL || err->empty();
}

static bool gpu_warmup(const SimParams& p, int block) {
  std::vector<float> scratch;
  double ms = 0.0;
  std::string err;
  return gpu_run_once(p, 1024, block, 1ULL, GPU_KERNEL_ONLY, scratch, &ms, &err);
}

#endif  // GBM_HAS_CUDA

// -----------------------------------------------------------------------------
//  Result rows
// -----------------------------------------------------------------------------

struct CorrectnessRow {
  std::string check;
  std::string backend;
  long long paths;
  int steps;
  uint64_t seed;
  double value;
  double expected;
  double rel_err;
  double z;
  std::string status;  // PASS / FAIL / SKIP
  std::string note;
};

// Column order must match the fprintf() in write_correctness_csv() exactly; the
// field count is asserted below so header/row drift cannot slip through.
static const char* K_CORRECTNESS_HEADER =
    "check,backend,paths,steps,seed,value,expected,rel_err,z_score,status,note";

static int count_csv_fields(const char* s, bool respect_quotes) {
  int n = 1;
  bool in_quotes = false;
  for (const char* q = s; *q; ++q) {
    if (respect_quotes && *q == '"') in_quotes = !in_quotes;
    else if (*q == ',' && !in_quotes) n++;
  }
  return n;
}

// Returns false if the file could not be written or if any row's field count
// disagrees with the header, so a malformed CSV fails the run instead of being
// silently produced.
static bool write_correctness_csv(const std::vector<CorrectnessRow>& rows) {
  FILE* f = std::fopen(path_in_out("correctness.csv").c_str(), "w");
  if (!f) {
    std::printf("[error] cannot write %s\n", path_in_out("correctness.csv").c_str());
    return false;
  }
  const int n_head = count_csv_fields(K_CORRECTNESS_HEADER, false);
  bool ok = true;
  std::fprintf(f, "%s\n", K_CORRECTNESS_HEADER);
  for (size_t i = 0; i < rows.size(); ++i) {
    const CorrectnessRow& r = rows[i];
    // The note field is free text and contains commas, so it is quoted and any
    // embedded double quote is doubled -- otherwise the file is not valid CSV.
    std::string note = r.note;
    std::string quoted;
    for (size_t k = 0; k < note.size(); ++k) {
      if (note[k] == '"') quoted += '"';
      quoted += note[k];
    }
    char line[4096];
    std::snprintf(line, sizeof(line), "%s,%s,%lld,%d,%llu,%.10g,%.10g,%.6g,%.6g,%s,\"%s\"",
                  r.check.c_str(), r.backend.c_str(), r.paths, r.steps,
                  (unsigned long long)r.seed, r.value, r.expected, r.rel_err, r.z,
                  r.status.c_str(), quoted.c_str());
    if (count_csv_fields(line, true) != n_head) {
      std::printf("[error] row/header CSV column mismatch: %d fields vs %d header columns\n",
                  count_csv_fields(line, true), n_head);
      ok = false;
    }
    std::fprintf(f, "%s\n", line);
  }
  std::fclose(f);
  return ok;
}

// -----------------------------------------------------------------------------
//  Correctness command
// -----------------------------------------------------------------------------

static const double GBM_Z_LIMIT = 4.0;  // ~1e-4 two-sided false-failure rate

static void add_check(std::vector<CorrectnessRow>& rows, bool ok, const std::string& check,
                      const std::string& backend, long long paths, int steps, uint64_t seed,
                      double value, double expected, double z, const std::string& note) {
  CorrectnessRow r;
  r.check = check;
  r.backend = backend;
  r.paths = paths;
  r.steps = steps;
  r.seed = seed;
  r.value = value;
  r.expected = expected;
  r.rel_err = (expected != 0.0) ? (value - expected) / expected : 0.0;
  r.z = z;
  r.status = ok ? "PASS" : "FAIL";
  r.note = note;
  rows.push_back(r);
}

static int cmd_verify(int argc, char** argv) {
  const long long n_paths = get_opt_ll(argc, argv, "--paths", 200000);
  const int n_seeds = (int)get_opt_ll(argc, argv, "--seeds", 3);
  const int steps = (int)get_opt_ll(argc, argv, "--steps", 252);
  g_out_dir = get_opt(argc, argv, "--out", "results");
  ensure_dir(g_out_dir);

  SimParams p;
  p.S0 = 100.0;
  p.mu = 0.05;
  p.sigma = 0.2;
  p.T = 1.0;
  p.steps = steps;

  const double E = analytic_mean(p);
  const double V = analytic_var(p);
  const double sem = se_mean(p, n_paths);
  const double sev = se_var(p, n_paths);

  std::vector<CorrectnessRow> rows;
  std::printf("== Correctness verification =====================================\n");
  std::printf("model   : GBM  S(t+dt)=S(t)*exp((mu-0.5*sigma^2)dt + sigma*sqrt(dt)*Z)\n");
  std::printf("params  : S0=%.3f mu=%.3f sigma=%.3f T=%.3f steps=%d paths=%lld seeds=%d\n", p.S0,
              p.mu, p.sigma, p.T, p.steps, n_paths, n_seeds);
  std::printf("analytic: E[S(T)]=%.6f  Var[S(T)]=%.6f  sd=%.6f\n", E, V, std::sqrt(V));
  std::printf("s.e.    : mean=%.6f  variance=%.6f  (z-limit |z| < %.1f)\n\n", sem, sev,
              GBM_Z_LIMIT);

  bool all_ok = true;
  int n_pass = 0, n_fail = 0, n_skip = 0;

  // --- per-seed distributional checks -------------------------------------
  for (int k = 0; k < n_seeds; ++k) {
    const uint64_t seed = 12345ULL + (uint64_t)k;
    std::vector<float> cpu_px;

    // ---- CPU
    double cpu_ms = cpu_simulate(p, n_paths, seed, cpu_px);
    Stats cs = compute_stats(cpu_px);
    {
      bool ok = (cs.n == n_paths) && (cs.n_nonfinite == 0) && (cs.n_nonpositive == 0);
      add_check(rows, ok, "output_valid", "cpu", n_paths, p.steps, seed, (double)cs.n,
                (double)n_paths, 0.0,
                "size + all finite + all > 0 (nonfinite=" + std::to_string(cs.n_nonfinite) +
                    ", nonpositive=" + std::to_string(cs.n_nonpositive) + ")");
      if (!ok) all_ok = false;
      ok ? n_pass++ : n_fail++;
    }
    {
      // Restrict the moment checks to the valid subset.
      double z = (cs.mean - E) / sem;
      bool ok = std::fabs(z) < GBM_Z_LIMIT;
      add_check(rows, ok, "mean_vs_analytic", "cpu", n_paths, p.steps, seed, cs.mean, E, z,
                "H0: sample mean == S0*exp(mu*T); tolerance = 4 analytical s.e.");
      if (!ok) all_ok = false;
      ok ? n_pass++ : n_fail++;
    }
    {
      double z = (sev > 0.0) ? (cs.var - V) / sev : 0.0;
      bool ok = (sev > 0.0) && (std::fabs(z) < GBM_Z_LIMIT);
      add_check(rows, ok, "variance_vs_analytic", "cpu", n_paths, p.steps, seed, cs.var, V, z,
                "log-normal central-4th-moment based s.e.");
      if (!ok) all_ok = false;
      ok ? n_pass++ : n_fail++;
    }
    std::printf("[cpu] seed=%llu  mean=%8.4f (E=%.4f, z=%+6.2f)  sd=%7.3f  time=%7.1f ms\n",
                (unsigned long long)seed, cs.mean, E, (cs.mean - E) / sem, cs.sd, cpu_ms);

#if GBM_HAS_CUDA
    // ---- GPU
    std::vector<float> gpu_px;
    double gpu_ms = -1.0;
    std::string err;
    bool launched = false;
    for (int rep = 0; rep < 3 && !launched; ++rep) {  // 3 attempts: warm-up then measure
      err.clear();
      launched = gpu_run_once(p, n_paths, 256, seed, GPU_KERNEL_ONLY, gpu_px, &gpu_ms, &err);
      if (!launched) std::printf("[warn] gpu attempt %d failed: %s\n", rep + 1, err.c_str());
    }
    if (!launched) {
      add_check(rows, false, "kernel_launch", "gpu", n_paths, p.steps, seed, 0, 0, 0, err);
      std::printf("[gpu] seed=%llu  LAUNCH FAILED: %s\n", (unsigned long long)seed, err.c_str());
      all_ok = false;
      n_fail++;
    } else {
      Stats gs = compute_stats(gpu_px);
      {
        bool ok = (gs.n == n_paths) && (gs.n_nonfinite == 0) && (gs.n_nonpositive == 0);
        add_check(rows, ok, "output_valid", "gpu", n_paths, p.steps, seed, (double)gs.n,
                  (double)n_paths, 0.0,
                  "size + all finite + all > 0 (nonfinite=" + std::to_string(gs.n_nonfinite) +
                      ", nonpositive=" + std::to_string(gs.n_nonpositive) + ")");
        if (!ok) all_ok = false;
        ok ? n_pass++ : n_fail++;
      }
      {
        double z = (gs.mean - E) / sem;
        bool ok = std::fabs(z) < GBM_Z_LIMIT;
        add_check(rows, ok, "mean_vs_analytic", "gpu", n_paths, p.steps, seed, gs.mean, E, z,
                  "H0: sample mean == S0*exp(mu*T); tolerance = 4 analytical s.e.");
        if (!ok) all_ok = false;
        ok ? n_pass++ : n_fail++;
      }
      {
        double z = (sev > 0.0) ? (gs.var - V) / sev : 0.0;
        bool ok = (sev > 0.0) && (std::fabs(z) < GBM_Z_LIMIT);
        add_check(rows, ok, "variance_vs_analytic", "gpu", n_paths, p.steps, seed, gs.var, V, z,
                  "log-normal central-4th-moment based s.e.");
        if (!ok) all_ok = false;
        ok ? n_pass++ : n_fail++;
      }
      {
        // Cross-implementation agreement on the MEAN only (different RNGs, so
        // element-by-element equality is not a meaningful requirement).
        double z = (cs.mean - gs.mean) / std::sqrt(2.0) / sem;
        bool ok = std::fabs(z) < GBM_Z_LIMIT;
        add_check(rows, ok, "mean_cpu_vs_gpu", "cpu_gpu", n_paths, p.steps, seed,
                  cs.mean - gs.mean, 0.0, z,
                  "two independent samples, s.e. = sqrt(2)*s.e.(mean); elementwise equality NOT required");
        if (!ok) all_ok = false;
        ok ? n_pass++ : n_fail++;
        add_check(rows, true, "elementwise_cpu_vs_gpu", "cpu_gpu", n_paths, p.steps, seed, 0, 0, 0,
                  "SKIPPED BY DESIGN: CPU uses mt19937_64+Box-Muller (double), GPU uses "
                  "Philox4x32 (float); the two draw different normal streams");
        n_skip++;
      }
      std::printf("[gpu] seed=%llu  mean=%8.4f (E=%.4f, z=%+6.2f)  sd=%7.3f  kernel=%7.3f ms\n",
                  (unsigned long long)seed, gs.mean, E, (gs.mean - E) / sem, gs.sd, gpu_ms);

      // ---- determinism: identical seed must reproduce identical output -----
      if (k == 0) {
        std::vector<float> again;
        double ms2 = 0.0;
        std::string e2;
        bool ok2 = gpu_run_once(p, n_paths, 256, seed, GPU_KERNEL_ONLY, again, &ms2, &e2);
        if (ok2) {
          double maxdiff = 0.0;
          for (size_t i = 0; i < again.size(); ++i)
            maxdiff = std::max(maxdiff, (double)std::fabs((double)again[i] - (double)gpu_px[i]));
          bool ok = (maxdiff == 0.0);
          add_check(rows, ok, "determinism_same_seed", "gpu", n_paths, p.steps, seed, maxdiff, 0.0,
                    0.0, "bitwise reproducibility for a fixed (seed, path count, block size)");
          if (!ok) all_ok = false;
          ok ? n_pass++ : n_fail++;
        }
        // Different seed -> different sample (guards against a constant stream).
        std::vector<float> other;
        double ms3 = 0.0;
        std::string e3;
        bool ok3 = gpu_run_once(p, n_paths, 256, seed + 7777ULL, GPU_KERNEL_ONLY, other, &ms3, &e3);
        if (ok3) {
          double d = 0.0;
          for (size_t i = 0; i < other.size(); ++i)
            d = std::max(d, (double)std::fabs((double)other[i] - (double)gpu_px[i]));
          bool ok = (d > 0.0);
          add_check(rows, ok, "different_seed_differs", "gpu", n_paths, p.steps, seed + 7777ULL, d,
                    0.0, 0.0, "max |dS| between two seeds must be > 0");
          if (!ok) all_ok = false;
          ok ? n_pass++ : n_fail++;
        }
      }
    }
#else
    (void)0;  // CPU-only build: no GPU checks in this loop
#endif
  }

  // --- edge cases (small, structural) --------------------------------------
  {
    const long long small = 1000;  // not a multiple of 256 -> 24 masked lanes
    SimParams ps = p;
    ps.steps = 252;
    std::vector<float> px;
    double ms = 0.0;
    std::string err;

#if GBM_HAS_CUDA
    bool ok = gpu_run_once(ps, small, 256, 999ULL, GPU_KERNEL_ONLY, px, &ms, &err);
    Stats s = compute_stats(px);
    double z = (s.mean - analytic_mean(ps)) / se_mean(ps, small);
    bool structural = ok && (s.n == small) && (s.n_nonfinite == 0) && (s.n_nonpositive == 0);
    bool statistical = structural && (std::fabs(z) < GBM_Z_LIMIT);
    add_check(rows, structural, "non_divisible_paths_structure", "gpu", small, 252, 999ULL,
              (double)s.n, (double)small, 0.0,
              "paths=1000, block=256 -> 24 threads must exit via the bounds guard");
    structural ? n_pass++ : n_fail++;
    add_check(rows, statistical, "non_divisible_paths_mean", "gpu", small, 252, 999ULL, s.mean,
              analytic_mean(ps), z, "loose mean check at N=1000");
    statistical ? n_pass++ : n_fail++;
    if (!structural || !statistical) all_ok = false;
    std::printf("[gpu] paths=1000 block=256 -> size=%lld finite_ok=%d mean_z=%+.2f\n", s.n,
                (int)structural, z);
#endif
    // CPU structural check on the same tiny case (independent of CUDA presence)
    std::vector<float> cpx;
    cpu_simulate(ps, small, 999ULL, cpx);
    Stats cs2 = compute_stats(cpx);
    bool okc = (cs2.n == small) && (cs2.n_nonfinite == 0) && (cs2.n_nonpositive == 0);
    add_check(rows, okc, "non_divisible_paths_structure", "cpu", small, 252, 999ULL, (double)cs2.n,
              (double)small, 0.0, "same case on the CPU baseline");
    okc ? n_pass++ : n_fail++;
    if (!okc) all_ok = false;
  }
  // single path
  {
    SimParams ps = p;
    std::vector<float> px;
    double ms = 0.0;
    std::string err;
#if GBM_HAS_CUDA
    bool ok = gpu_run_once(ps, 1, 256, 7ULL, GPU_KERNEL_ONLY, px, &ms, &err);
    bool good = ok && px.size() == 1 && std::isfinite((double)px[0]) && px[0] > 0.0f;
    add_check(rows, good, "single_path", "gpu", 1, p.steps, 7ULL, good ? (double)px[0] : 0.0, 0.0,
              0.0, "paths=1, block=256: 255 inactive threads");
    good ? n_pass++ : n_fail++;
    if (!good) all_ok = false;
#else
    SimParams ps2 = ps;
    std::vector<float> c1;
    cpu_simulate(ps2, 1, 7ULL, c1);
    bool good = c1.size() == 1 && std::isfinite((double)c1[0]) && c1[0] > 0.0f;
    add_check(rows, good, "single_path", "cpu", 1, p.steps, 7ULL, (double)c1[0], 0.0, 0.0,
              "paths=1 on the CPU baseline");
    good ? n_pass++ : n_fail++;
    if (!good) all_ok = false;
#endif
  }
#if GBM_HAS_CUDA
  // zero paths must not crash
  {
    std::vector<float> px;
    double ms = -1.0;
    std::string err;
    bool ok = gpu_run_once(p, 0, 256, 1ULL, GPU_KERNEL_ONLY, px, &ms, &err);
    bool good = ok && px.empty();
    add_check(rows, good, "zero_paths", "gpu", 0, p.steps, 1ULL, 0, 0, 0,
              "paths=0 -> no launch, empty output, no crash");
    good ? n_pass++ : n_fail++;
    if (!good) all_ok = false;
  }
#endif
  // step-count invariance: the terminal distribution does not depend on `steps`
  {
    const long long n = std::max<long long>(50000, n_paths / 4);
    const int step_list[3] = {1, 21, 252};
    for (int si = 0; si < 3; ++si) {
      SimParams ps = p;
      ps.steps = step_list[si];
      std::vector<float> px;
      double ms = -1.0;
      std::string err;
      bool ok = true;
#if GBM_HAS_CUDA
      ok = gpu_run_once(ps, n, 256, 4242ULL, GPU_KERNEL_ONLY, px, &ms, &err);
#else
      cpu_simulate(ps, n, 4242ULL, px);
#endif
      Stats s = compute_stats(px);
      double z = (s.mean - analytic_mean(ps)) / se_mean(ps, n);
      bool good = ok && (std::fabs(z) < GBM_Z_LIMIT) && (s.n_nonfinite == 0) &&
                  (s.n_nonpositive == 0);
      add_check(rows, good, "steps_invariance", GBM_HAS_CUDA ? "gpu" : "cpu", n, ps.steps, 4242ULL,
                s.mean, analytic_mean(ps), z, "E[S(T)] must not depend on the number of Euler steps");
      good ? n_pass++ : n_fail++;
      if (!good) all_ok = false;
    }
  }

  const bool csv_ok = write_correctness_csv(rows);
  if (!csv_ok) {
    all_ok = false;
    n_fail++;
  }

  std::printf("\n-- summary: %d PASS, %d FAIL, %d SKIPPED (by design) --\n", n_pass, n_fail, n_skip);
  std::printf("-- wrote %s --\n", path_in_out("correctness.csv").c_str());
  if (!GBM_HAS_CUDA)
    std::printf("NOTE: built with -DGBM_CPU_ONLY, so every GPU check was not executed.\n");
  std::printf(all_ok ? "RESULT: ALL CHECKS PASSED\n" : "RESULT: FAILURES PRESENT\n");
  return all_ok ? 0 : 1;
}

// -----------------------------------------------------------------------------
//  Benchmark command: CPU vs GPU over sizes, and GPU over block sizes
// -----------------------------------------------------------------------------

struct BenchSizeRow {
  long long paths;
  int steps;
  int block;
  double cpu_ms;
  double gpu_kernel_ms;
  double gpu_e2e_ms;
  double speedup_kernel;
  double speedup_e2e;
  double gpu_paths_per_sec;
  int cpu_reps;
  int gpu_reps;
};

struct BenchBlockRow {
  long long paths;
  int steps;
  int block;
  double gpu_kernel_ms;
  double gpu_e2e_ms;
  double gpu_paths_per_sec;
  double paths_per_sec_per_step;  // terminal-step throughput normalised by steps
  int reps;
};

static int cmd_bench(int argc, char** argv) {
  g_out_dir = get_opt(argc, argv, "--out", "results");
  ensure_dir(g_out_dir);

  const std::string sizes_s = get_opt(argc, argv, "--sizes", "10000,100000,1000000");
  const std::string blocks_s = get_opt(argc, argv, "--blocks", "64,128,256,512");
  const int steps = (int)get_opt_ll(argc, argv, "--steps", 252);
  const int reps = (int)get_opt_ll(argc, argv, "--reps", 5);
  const int cpu_reps_cfg = (int)get_opt_ll(argc, argv, "--cpu-reps", 3);
  const int ref_block = (int)get_opt_ll(argc, argv, "--ref-block", 256);

  std::vector<long long> sizes = parse_int_list(sizes_s);
  std::vector<long long> blocks = parse_int_list(blocks_s);
  if (sizes.empty() || blocks.empty()) {
    std::printf("empty --sizes/--blocks\n");
    return 2;
  }

  SimParams p;
  p.S0 = 100.0;
  p.mu = 0.05;
  p.sigma = 0.2;
  p.T = 1.0;
  p.steps = steps;

  std::printf("== Benchmarks ===================================================\n");
  std::printf("steps=%d  gpu reps=%d  cpu reps=%d  block sweep=%s  sizes=%s\n", steps, reps,
              cpu_reps_cfg, blocks_s.c_str(), sizes_s.c_str());
  std::printf("method: warm-up run, then median of %d CUDA-event kernel-only timings; GPU e2e = "
              "median of %d runs incl. cudaMalloc/launch/D2H copy/cudaFree\n\n",
              reps, reps);

#if GBM_HAS_CUDA
  gpu_warmup(p, ref_block);  // size/arch warm-up before any timed run
#endif

  std::vector<BenchSizeRow> size_rows;
  std::vector<BenchBlockRow> block_rows;

  for (size_t si = 0; si < sizes.size(); ++si) {
    const long long n = sizes[si];
    const int cpu_reps = std::max(1, cpu_reps_cfg);

    // ---------------- CPU baseline -------------------------------------
    std::vector<double> cpu_times;
    std::vector<float> cpu_px;
    Stats cpu_stats;
    for (int r = 0; r < cpu_reps; ++r) {
      const uint64_t seed = 2024ULL + (uint64_t)r;
      cpu_times.push_back(cpu_simulate(p, n, seed, cpu_px));
      cpu_stats = compute_stats(cpu_px);
    }
    const double cpu_ms = median(cpu_times);

    std::printf("paths=%-9lld  CPU  median=%9.3f ms  (reps=%d, mean=%.3f, sd=%.3f)\n", n, cpu_ms,
                cpu_reps, cpu_stats.mean, cpu_stats.sd);

    // The CPU row is recorded in every build; GPU columns stay at -1/0 when no
    // CUDA device is compiled in, so the CSV schema is always the same.
    BenchSizeRow row;
    row.paths = n;
    row.steps = steps;
    row.block = ref_block;
    row.cpu_ms = cpu_ms;
    row.gpu_kernel_ms = -1.0;
    row.gpu_e2e_ms = -1.0;
    row.speedup_kernel = 0.0;
    row.speedup_e2e = 0.0;
    row.gpu_paths_per_sec = 0.0;
    row.cpu_reps = cpu_reps;
    row.gpu_reps = 0;

#if GBM_HAS_CUDA
    // ---------------- GPU: kernel-only + end-to-end --------------------
    std::vector<double> kern_times, e2e_times;
    std::vector<float> gpu_px;
    std::string err;
    for (int r = 0; r < reps; ++r) {
      const uint64_t seed = 2024ULL;  // fixed seed -> identical work every rep
      double ms = 0.0;
      err.clear();
      if (!gpu_run_once(p, n, ref_block, seed, GPU_KERNEL_ONLY, gpu_px, &ms, &err)) {
        std::printf("  [error] kernel run failed: %s\n", err.c_str());
        break;
      }
      kern_times.push_back(ms);
    }
    for (int r = 0; r < reps; ++r) {
      double ms = 0.0;
      std::vector<float> tmp;
      err.clear();
      if (!gpu_run_once(p, n, ref_block, 2024ULL, GPU_END_TO_END, tmp, &ms, &err)) {
        std::printf("  [error] e2e run failed: %s\n", err.c_str());
        break;
      }
      e2e_times.push_back(ms);
    }
    const double gk = median(kern_times);
    const double ge = median(e2e_times);

    row.gpu_kernel_ms = gk;
    row.gpu_e2e_ms = ge;
    row.speedup_kernel = (gk > 0.0) ? cpu_ms / gk : 0.0;
    row.speedup_e2e = (ge > 0.0) ? cpu_ms / ge : 0.0;
    row.gpu_paths_per_sec = (gk > 0.0) ? (double)n / (gk / 1000.0) : 0.0;
    row.gpu_reps = reps;
    std::printf("paths=%-9lld  GPU  kernel=%9.3f ms  e2e=%9.3f ms  kernel speedup=%7.2fx  "
                "e2e speedup=%7.2fx\n\n",
                n, gk, ge, row.speedup_kernel, row.speedup_e2e);

    // ---------------- GPU: block-size sweep at this size ---------------
    for (size_t bi = 0; bi < blocks.size(); ++bi) {
      const int b = (int)blocks[bi];
      std::vector<double> kt, et;
      for (int r = 0; r < reps; ++r) {
        double ms = 0.0;
        err.clear();
        if (!gpu_run_once(p, n, b, 2024ULL, GPU_KERNEL_ONLY, gpu_px, &ms, &err)) break;
        kt.push_back(ms);
      }
      for (int r = 0; r < reps; ++r) {
        double ms = 0.0;
        std::vector<float> tmp;
        err.clear();
        if (!gpu_run_once(p, n, b, 2024ULL, GPU_END_TO_END, tmp, &ms, &err)) break;
        et.push_back(ms);
      }
      if (kt.empty()) continue;
      BenchBlockRow br;
      br.paths = n;
      br.steps = steps;
      br.block = b;
      br.gpu_kernel_ms = median(kt);
      br.gpu_e2e_ms = et.empty() ? -1.0 : median(et);
      br.gpu_paths_per_sec = br.gpu_kernel_ms > 0.0 ? (double)n / (br.gpu_kernel_ms / 1000.0) : 0.0;
      br.paths_per_sec_per_step = br.gpu_paths_per_sec / (double)steps;
      br.reps = (int)kt.size();
      block_rows.push_back(br);
      std::printf("    block=%-4d kernel=%9.3f ms  e2e=%9.3f ms  %10.3f Mpath/s\n", b,
                  br.gpu_kernel_ms, br.gpu_e2e_ms, br.gpu_paths_per_sec / 1e6);
    }
    std::printf("\n");
#else
    std::printf("  (CPU-only build: no GPU benchmarks for this size)\n\n");
#endif
    size_rows.push_back(row);
  }

  // ---------------- write CSVs ------------------------------------------
  {
    FILE* f = std::fopen(path_in_out("benchmark_sizes.csv").c_str(), "w");
    if (f) {
      std::fprintf(f,
                   "paths,steps,block,cpu_ms,gpu_kernel_ms,gpu_e2e_ms,speedup_kernel,speedup_e2e,"
                   "gpu_paths_per_sec,cpu_reps,gpu_reps\n");
      for (size_t i = 0; i < size_rows.size(); ++i) {
        const BenchSizeRow& r = size_rows[i];
        std::fprintf(f, "%lld,%d,%d,%.6f,%.6f,%.6f,%.6f,%.6f,%.1f,%d,%d\n", r.paths, r.steps,
                     r.block, r.cpu_ms, r.gpu_kernel_ms, r.gpu_e2e_ms, r.speedup_kernel,
                     r.speedup_e2e, r.gpu_paths_per_sec, r.cpu_reps, r.gpu_reps);
      }
      std::fclose(f);
      std::printf("wrote %s\n", path_in_out("benchmark_sizes.csv").c_str());
    }
    FILE* g = std::fopen(path_in_out("benchmark_blocks.csv").c_str(), "w");
    if (g) {
      std::fprintf(g,
                   "paths,steps,block,gpu_kernel_ms,gpu_e2e_ms,gpu_paths_per_sec,"
                   "paths_per_sec_per_step,reps\n");
      for (size_t i = 0; i < block_rows.size(); ++i) {
        const BenchBlockRow& r = block_rows[i];
        std::fprintf(g, "%lld,%d,%d,%.6f,%.6f,%.1f,%.3f,%d\n", r.paths, r.steps, r.block,
                     r.gpu_kernel_ms, r.gpu_e2e_ms, r.gpu_paths_per_sec, r.paths_per_sec_per_step,
                     r.reps);
      }
      std::fclose(g);
      std::printf("wrote %s\n", path_in_out("benchmark_blocks.csv").c_str());
    }
  }
  return 0;
}

// -----------------------------------------------------------------------------
//  RNG / launch-overhead experiment
//  Sweeping the step count at a fixed path count gives  t(steps) = a + b*steps.
//  `b` is the marginal per-step cost (normal draws + exp + multiply) and `a`
//  captures the fixed per-launch cost: kernel launch + `curand_init` for every
//  thread. The fit is produced by ordinary least squares below.
// -----------------------------------------------------------------------------
static int cmd_rng(int argc, char** argv) {
  g_out_dir = get_opt(argc, argv, "--out", "results");
  ensure_dir(g_out_dir);

  const long long n = get_opt_ll(argc, argv, "--paths", 1000000);
  const int block = (int)get_opt_ll(argc, argv, "--block", 256);
  const int reps = (int)get_opt_ll(argc, argv, "--reps", 5);
  std::vector<long long> steps_list = parse_int_list(get_opt(argc, argv, "--steps", "1,4,16,64,252"));

  SimParams p;
  p.S0 = 100.0;
  p.mu = 0.05;
  p.sigma = 0.2;
  p.T = 1.0;

  std::printf("== RNG / launch overhead study ==================================\n");
  std::printf("paths=%lld block=%d reps=%d steps=%s\n\n", n, block, reps,
              join_csv(steps_list).c_str());

#if GBM_HAS_CUDA
  gpu_warmup(p, block);
#endif

  std::vector<double> xs, ys;
  FILE* f = std::fopen(path_in_out("rng_overhead.csv").c_str(), "w");
  if (f)
    std::fprintf(f, "paths,block,steps,kernel_ms,us_per_step,ns_per_path_step,gpu_paths_per_sec\n");

  for (size_t i = 0; i < steps_list.size(); ++i) {
    const int st = (int)steps_list[i];
    if (st < 1) continue;
    SimParams ps = p;
    ps.steps = st;

    double ms = -1.0;
#if GBM_HAS_CUDA
    std::vector<double> times;
    std::vector<float> px;
    std::string err;
    for (int r = 0; r < reps; ++r) {
      double t = 0.0;
      err.clear();
      if (!gpu_run_once(ps, n, block, 2024ULL, GPU_KERNEL_ONLY, px, &t, &err)) {
        std::printf("  [error] %s\n", err.c_str());
        break;
      }
      times.push_back(t);
    }
    if (!times.empty()) ms = median(times);
#else
    std::vector<float> px;
    ms = cpu_simulate(ps, n, 2024ULL, px);
#endif
    if (ms < 0.0) continue;
    xs.push_back((double)st);
    ys.push_back(ms);
    std::printf("  steps=%-4d kernel=%9.4f ms  %8.4f us/step  %7.3f ns/path/step  %10.1f Mpath/s\n",
                st, ms, ms * 1000.0 / (double)st,
                ms * 1e6 / ((double)st * (double)n), (double)n / (ms / 1000.0) / 1e6);
    if (f)
      std::fprintf(f, "%lld,%d,%d,%.6f,%.4f,%.4f,%.1f\n", n, block, st, ms,
                   ms * 1000.0 / (double)st, ms * 1e6 / ((double)st * (double)n),
                   (double)n / (ms / 1000.0));
  }
  if (f) std::fclose(f);

  // Ordinary least squares fit  t = a + b*steps
  if (xs.size() >= 2) {
    const double m = (double)xs.size();
    double sx = 0, sy = 0, sxx = 0, sxy = 0;
    for (size_t i = 0; i < xs.size(); ++i) {
      sx += xs[i];
      sy += ys[i];
      sxx += xs[i] * xs[i];
      sxy += xs[i] * ys[i];
    }
    const double denom = m * sxx - sx * sx;
    const double b = (denom != 0.0) ? (m * sxy - sx * sy) / denom : 0.0;
    const double a = (sy - b * sx) / m;
    double ss_tot = 0, ss_res = 0;
    const double ybar = sy / m;
    for (size_t i = 0; i < xs.size(); ++i) {
      const double fit = a + b * xs[i];
      ss_tot += (ys[i] - ybar) * (ys[i] - ybar);
      ss_res += (ys[i] - fit) * (ys[i] - fit);
    }
    const double r2 = (ss_tot > 0.0) ? 1.0 - ss_res / ss_tot : 1.0;
    const double fixed_frac = (a + b * 252.0 != 0.0) ? a / (a + b * 252.0) : 0.0;

    char buf[1024];
#if GBM_HAS_CUDA
    const char* src_label = "kernel-only CUDA-event timings";
    const char* fixed_label = "fixed per-launch cost (kernel launch + curand_init for all";
    const char* per_path_label = "per-path curand_init+launch cost";
#else
    const char* src_label = "CPU wall-clock timings (CPU-only build)";
    const char* fixed_label = "fixed per-run cost (RNG seeding for all";
    const char* per_path_label = "per-path RNG-seeding cost";
#endif
    std::snprintf(buf, sizeof(buf),
                  "OLS fit over the step sweep (%s)\n"
                  "  t_ms(steps) = a + b*steps\n"
                  "  a = %.6f ms   %s %lld paths)\n"
                  "  b = %.8f ms   marginal cost of one extra step for all %lld paths\n"
                  "  R^2 = %.6f  (points = %d)\n"
                  "  at 252 steps: a + 252*b = %.6f ms, of which %.1f%% is fixed setup overhead\n"
                  "  %s ~ a / n = %.4f ns\n",
                  src_label, a, fixed_label, n, b, n, r2, (int)xs.size(), a + 252.0 * b,
                  100.0 * fixed_frac, per_path_label, a * 1e6 / (double)n);
    std::printf("\n%s", buf);
    FILE* ff = std::fopen(path_in_out("rng_overhead_fit.txt").c_str(), "w");
    if (ff) {
      std::fprintf(ff, "%s", buf);
      std::fclose(ff);
      std::printf("wrote %s (and rng_overhead.csv)\n", path_in_out("rng_overhead_fit.txt").c_str());
    }
  }
  return 0;
}

// -----------------------------------------------------------------------------
//  Device / environment information
// -----------------------------------------------------------------------------
static int cmd_info(int argc, char** argv) {
  g_out_dir = get_opt(argc, argv, "--out", "results");
  ensure_dir(g_out_dir);
  FILE* f = std::fopen(path_in_out("environment_cuda.txt").c_str(), "w");
  std::string log;

#if GBM_HAS_CUDA
  int dev_count = 0;
  cudaError_t rc = cudaGetDeviceCount(&dev_count);
  char buf[4096];
  std::snprintf(buf, sizeof(buf), "cudaGetDeviceCount -> %d (%s)\n", dev_count,
                cudaGetErrorString(rc));
  log += buf;
  if (rc == cudaSuccess && dev_count > 0) {
    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);
    int rt_ver = 0, drv_ver = 0;
    cudaRuntimeGetVersion(&rt_ver);
    cudaDriverGetVersion(&drv_ver);
    std::snprintf(buf, sizeof(buf),
                  "device                        : %s\n"
                  "compute capability            : %d.%d\n"
                  "multiprocessors (SMs)         : %d\n"
                  "max threads per SM            : %d\n"
                  "max threads per block         : %d\n"
                  "shared memory per block       : %zu bytes\n"
                  "warp size                     : %d\n"
                  "registers per SM              : %d\n"
                  "global memory                 : %.2f GiB\n"
                  "memory bus width              : %d bit\n"
                  "memory clock rate             : %.0f MHz\n"
                  "L2 cache size                 : %d bytes\n"
                  "concurrent kernels            : %d\n"
                  "cudaRuntimeGetVersion         : %d\n"
                  "cudaDriverGetVersion          : %d\n"
                  "compiled with nvcc            : %d.%d\n",
                  prop.name, prop.major, prop.minor, prop.multiProcessorCount,
                  prop.maxThreadsPerMultiProcessor, prop.maxThreadsPerBlock,
                  (size_t)prop.sharedMemPerBlock, prop.warpSize, prop.regsPerMultiprocessor,
                  (double)prop.totalGlobalMem / 1073741824.0, prop.memoryBusWidth,
                  (double)prop.memoryClockRate / 1000.0, prop.l2CacheSize, prop.concurrentKernels,
                  rt_ver, drv_ver,
#if defined(__CUDACC__)
                  __CUDACC_VER_MAJOR__, __CUDACC_VER_MINOR__
#else
                  0, 0
#endif
    );
    log += buf;
  }
#else
  log += "built with -DGBM_CPU_ONLY: CUDA support not compiled in\n";
#endif
  log += "host: see nvidia-smi / lscpu output captured by the notebook\n";

  std::printf("%s", log.c_str());
  if (f) {
    std::fprintf(f, "%s", log.c_str());
    std::fclose(f);
    std::printf("wrote %s\n", path_in_out("environment_cuda.txt").c_str());
  }
  return 0;
}

// -----------------------------------------------------------------------------
//  Smoke test: fast correctness + tiny benchmark. Run this before the full
//  benchmark matrix so a broken build fails in seconds instead of minutes.
// -----------------------------------------------------------------------------
static int cmd_smoke(int argc, char** argv) {
  g_out_dir = get_opt(argc, argv, "--out", "results");
  SimParams p;
  p.S0 = 100.0;
  p.mu = 0.05;
  p.sigma = 0.2;
  p.T = 1.0;
  p.steps = 252;
  const long long n = 100000;
  const double E = analytic_mean(p);
  const double sem = se_mean(p, n);
  bool ok = true;

  std::printf("== Smoke test ===================================================\n");
  std::vector<float> px;
  double t_cpu = cpu_simulate(p, n, 1ULL, px);
  Stats cs = compute_stats(px);
  double zc = (cs.mean - E) / sem;
  std::printf("cpu: %lld paths, %6.1f ms, mean=%.4f (E=%.4f, z=%+.2f)\n", n, t_cpu, cs.mean, E, zc);
  ok = ok && std::fabs(zc) < GBM_Z_LIMIT && cs.n_nonfinite == 0 && cs.n_nonpositive == 0;

#if GBM_HAS_CUDA
  gpu_warmup(p, 128);
  double ms = -1.0, e2e = -1.0;
  std::string err;
  std::vector<float> gx, gy;
  bool launched = gpu_run_once(p, n, 128, 1ULL, GPU_KERNEL_ONLY, gx, &ms, &err);
  Stats gs = compute_stats(gx);
  double zg = (gs.mean - E) / sem;
  gpu_run_once(p, n, 128, 1ULL, GPU_END_TO_END, gy, &e2e, &err);
  std::printf("gpu: %lld paths, block=128, kernel=%7.3f ms, e2e=%7.3f ms, mean=%.4f (z=%+.2f)\n", n,
              ms, e2e, gs.mean, zg);
  std::printf("speedup (kernel): %.2fx\n", (ms > 0.0) ? t_cpu / ms : 0.0);
  ok = ok && launched && std::fabs(zg) < GBM_Z_LIMIT && gs.n_nonfinite == 0 &&
       gs.n_nonpositive == 0;
#else
  std::printf("gpu: SKIPPED (built with -DGBM_CPU_ONLY)\n");
#endif
  std::printf(ok ? "SMOKE OK\n" : "SMOKE FAILED\n");
  return ok ? 0 : 1;
}

static void usage() {
  std::printf(
      "Monte Carlo GBM (CUDA) -- commands:\n"
      "  info    [--out DIR]\n"
      "  smoke   [--out DIR]\n"
      "  verify  [--paths N] [--seeds K] [--steps S] [--out DIR]\n"
      "  bench   [--sizes a,b,c] [--blocks a,b,c] [--steps S] [--reps R] [--cpu-reps R]\n"
      "          [--ref-block B] [--out DIR]\n"
      "  rng     [--paths N] [--block B] [--steps a,b,c] [--reps R] [--out DIR]\n");
}

int main(int argc, char** argv) {
  if (argc < 2) {
    usage();
    return 2;
  }
  const std::string cmd = argv[1];
  if (cmd == "verify") return cmd_verify(argc, argv);
  if (cmd == "bench") return cmd_bench(argc, argv);
  if (cmd == "rng") return cmd_rng(argc, argv);
  if (cmd == "info") return cmd_info(argc, argv);
  if (cmd == "smoke") return cmd_smoke(argc, argv);
  usage();
  return 2;
}

'''

# The raw literal above carries one extra newline at each end (it is delimited by
# r'''...''' on their own lines). Strip exactly those, so the file written here is
# byte-identical to the src/gbm_montecarlo.cu in the repository - this is the copy
# that gets archived as "the source that produced these numbers".
SOURCE = SOURCE.strip("\n") + "\n"
SRC = pathlib.Path("src/gbm_montecarlo.cu")
SRC.write_text(SOURCE, encoding="utf-8")
print("wrote %s: %d lines, %d bytes" % (SRC, len(SOURCE.splitlines()), len(SOURCE.encode())))

CMD = [NVCC, "-O3", "-std=c++17", "-arch=%s" % ARCH, "-lineinfo", "-Xptxas", "-v",
       "-o", "build/gbm", "src/gbm_montecarlo.cu", "-lcurand"] + EXTRA_FLAGS
t0 = time.time()
p = subprocess.run(CMD, capture_output=True, text=True)
compile_log = "$ " + " ".join(CMD) + "\n\n" + p.stdout + p.stderr
pathlib.Path("results/compile_log.txt").write_text(compile_log)
print("compile rc = %d in %.1f s" % (p.returncode, time.time() - t0))
print(p.stdout[-6000:])
print(p.stderr[-6000:])
assert p.returncode == 0, "nvcc failed - see results/compile_log.txt and the output above"
assert pathlib.Path("build/gbm").exists(), "no binary produced"
print("\n[ptxas resource usage is recorded in results/compile_log.txt]")

banner("device information")
info = run(["./build/gbm", "info", "--out", "results"], quiet=False)

banner("smoke test (must pass before the benchmark matrix)")
smoke = run(["./build/gbm", "smoke", "--out", "results"])
pathlib.Path("results/smoke_stdout.txt").write_text(smoke.stdout + smoke.stderr)
assert smoke.returncode == 0, "smoke test failed"

banner("numerical correctness suite")
verify = run(["./build/gbm", "verify", "--paths", "200000", "--seeds", "3",
              "--steps", "252", "--out", "results"])
pathlib.Path("results/verify_stdout.txt").write_text(verify.stdout + verify.stderr)
print("\nverify exit code:", verify.returncode)
print("checks written to results/correctness.csv")

In [ ]:
# ========================= 3. Benchmarks (CSV) ==============================
import pathlib, time

banner("benchmark matrix: sizes x block sizes")
t0 = time.time()
bench = run(["./build/gbm", "bench",
             "--sizes", "10000,100000,1000000",
             "--blocks", "64,128,256,512",
             "--steps", "252",
             "--reps", "5",
             "--cpu-reps", "3",
             "--ref-block", "256",
             "--out", "results"])
pathlib.Path("results/bench_stdout.txt").write_text(bench.stdout + bench.stderr)
print("\nbench wall time: %.1f s (rc=%d)" % (time.time() - t0, bench.returncode))
assert bench.returncode == 0, "benchmark run failed"

banner("RNG setup / launch-overhead study (step sweep + OLS fit)")
rng = run(["./build/gbm", "rng",
           "--paths", "1000000",
           "--block", "256",
           "--steps", "1,4,16,64,252",
           "--reps", "5",
           "--out", "results"])
pathlib.Path("results/rng_stdout.txt").write_text(rng.stdout + rng.stderr)

print("\nfiles produced:")
for f in sorted(pathlib.Path("results").glob("*")):
    print("  %-34s %9d bytes" % (f.name, f.stat().st_size))

In [ ]:
# ============= 4. Plots, profiling attempt, result summary ==================
import os, pathlib, shutil, subprocess, textwrap
os.environ.setdefault("MPLBACKEND", "Agg")

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RES = pathlib.Path("results")
sizes = pd.read_csv(RES / "benchmark_sizes.csv")
blocks = pd.read_csv(RES / "benchmark_blocks.csv")
sizes_g = sizes[sizes.gpu_kernel_ms > 0].copy()
blocks_g = blocks[blocks.gpu_kernel_ms > 0].copy()

# ---- plot 1: CPU vs GPU time over simulation size -------------------------
fig, ax = plt.subplots(figsize=(8, 5), dpi=140)
ax.plot(sizes_g.paths, sizes_g.cpu_ms, "o-", lw=2, color="#c0392b",
        label="CPU baseline, 1 thread (mt19937_64 + Box-Muller)")
ax.plot(sizes_g.paths, sizes_g.gpu_kernel_ms, "s-", lw=2, color="#1f6feb",
        label="GPU kernel only (CUDA events)")
ax.plot(sizes_g.paths, sizes_g.gpu_e2e_ms, "^--", lw=2, color="#2e8b57",
        label="GPU end-to-end (malloc + launch + D2H copy + free)")
for _, r in sizes_g.iterrows():
    ax.annotate("%.1fx" % r.speedup_kernel, (r.paths, r.gpu_kernel_ms),
                textcoords="offset points", xytext=(6, -14), fontsize=8, color="#1f6feb")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Monte Carlo paths (252 steps, T = 1 year, S0 = 100, mu = 0.05, sigma = 0.20)")
ax.set_ylabel("wall time [ms]")
ax.set_title("Monte Carlo GBM: CPU vs custom CUDA kernel")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(RES / "plot_cpu_vs_gpu_time.png")
plt.close(fig)

# ---- plot 2: block-size sweep --------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), dpi=140)
for n, grp in blocks_g.groupby("paths"):
    g = grp.sort_values("block")
    axes[0].plot(g.block, g.gpu_kernel_ms, "o-", label="%s paths" % format(int(n), ","))
    axes[1].plot(g.block, g.gpu_paths_per_sec / 1e6, "o-", label="%s paths" % format(int(n), ","))
axes[0].set_xlabel("CUDA block size [threads]")
axes[0].set_ylabel("kernel time [ms]  (median of 5, CUDA events)")
axes[0].set_title("Kernel time vs block size, 252 steps")
axes[0].set_xscale("log", base=2)
axes[0].set_xticks(sorted(blocks_g.block.unique()))
axes[0].get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
axes[1].set_xlabel("CUDA block size [threads]")
axes[1].set_ylabel("throughput [Million paths/s]")
axes[1].set_title("Terminal-price throughput vs block size")
axes[1].set_xscale("log", base=2)
axes[1].set_xticks(sorted(blocks_g.block.unique()))
axes[1].get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
for a in axes:
    a.grid(True, alpha=0.3)
    a.legend(fontsize=8)
fig.tight_layout()
fig.savefig(RES / "plot_block_sizes.png")
plt.close(fig)

# ---- plot 3: speedup -----------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4), dpi=140)
x = np.arange(len(sizes_g))
w = 0.36
ax.bar(x - w / 2, sizes_g.speedup_kernel, w, label="kernel-only speedup", color="#1f6feb")
ax.bar(x + w / 2, sizes_g.speedup_e2e, w, label="end-to-end speedup", color="#2e8b57")
ax.set_xticks(x)
ax.set_xticklabels([format(int(p), ",") for p in sizes_g.paths])
ax.set_xlabel("Monte Carlo paths")
ax.set_ylabel("CPU time / GPU time  [x]")
ax.set_title("Speedup over the single-threaded CPU baseline (252 steps)")
ax.axhline(1.0, color="k", lw=1, ls=":")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(RES / "plot_speedup.png")
plt.close(fig)
print("wrote results/plot_cpu_vs_gpu_time.png, plot_block_sizes.png, plot_speedup.png")

# ---- profiling -----------------------------------------------------------
# A ladder that runs entirely inside this Kaggle runtime: use Nsight if the image
# ships it AND the container permits it, otherwise fall back to the strongest
# evidence that does work here -- a measured CUDA-event run of the real kernel,
# static kernel statistics read out of the built cubin (cuobjdump / ptxas), the
# occupancy that follows from them, and a sampled nvidia-smi telemetry trace.
# This section never touches the simulation: it reads the cubin, its compile log
# and the device properties, and it writes into results/profile/ so the benchmark
# CSVs written by cell 3 are left exactly as they are.
import re, threading, time

KERNEL = "gbm_terminal_price_kernel"
REP = (1000000, 256, 252, 3)  # paths, block, steps, reps - the representative run
PROFILE_DIR = pathlib.Path("results/profile")
PROFILE_DIR.mkdir(parents=True, exist_ok=True)


def find_tool(name):
    p = shutil.which(name)
    if p:
        return p
    for d in ("/usr/local/cuda/bin", "/opt/nvidia/nsight-systems/bin",
              "/opt/nvidia/nsight-compute/bin", "/usr/local/bin", "/usr/bin"):
        cand = pathlib.Path(d) / name
        if cand.exists():
            return str(cand)
    return None


def run_tool(label, cmd, timeout=900, cap=6000):
    '''Run a tool, echo it, return (text, returncode). Never raises.'''
    print("\n$ " + " ".join(cmd), flush=True)
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        out = (p.stdout or "") + (p.stderr or "")
        print(out[-cap:])
        return out, p.returncode
    except subprocess.TimeoutExpired:
        msg = "%s: timed out after %d s\n" % (label, timeout)
        print(msg)
        return msg, -1
    except OSError as exc:
        msg = "%s: could not be executed (%s)\n" % (label, exc)
        print(msg)
        return msg, -1


# ---- pure helpers (no GPU state, so they can be unit-tested off-device) -----
def _num(text):
    '''"1 590 MHz" / "45 %" / "25.30 W" -> float, or None when unsupported.'''
    t = text.strip()
    for unit in ("%", "MHz", "W", "MiB", "C"):
        t = t.replace(unit, "")
    try:
        return float(t.strip())
    except ValueError:
        return None


def _smi_ts(text):
    '''`2026/10/15 09:12:00.150` -> epoch seconds, keeping the milliseconds.

    time.strptime drops the fractional part, so it is parsed separately: the
    sampler runs at 50 ms, and without it every sample would land in the same
    second and the idle/busy split would collapse.
    '''
    t = text.strip()
    frac = 0.0
    if "." in t:
        t, _, us = t.partition(".")
        frac = float("0." + us)
    try:
        return time.mktime(time.strptime(t, "%Y/%m/%d %H:%M:%S")) + frac
    except ValueError:
        return None


def parse_ptxas_v(text):
    '''Register / smem / spill usage out of the nvcc -Xptxas -v compile log.'''
    info = {}
    for key, pat in (("registers_per_thread", r"Used\s+(\d+)\s+registers"),
                     ("static_smem_bytes", r"(\d+)\s+bytes\s+smem"),
                     ("spill_stores_bytes", r"(\d+)\s+bytes\s+spill stores"),
                     ("spill_loads_bytes", r"(\d+)\s+bytes\s+spill loads"),
                     # CUDA 11+ prints "0 bytes stack frame"; older ptxas printed
                     # "0 bytes cumulative stack size" - accept either.
                     ("stack_bytes", r"(\d+)\s+bytes\s+(?:stack frame|cumulative stack size)")):
        m = re.search(pat, text)
        if m:
            info[key] = int(m.group(1))
    return info


def parse_res_usage(text):
    '''REG/STACK/SHARED/LOCAL/... from `cuobjdump -res-usage`.

    Tolerates both `REG:26` and `REG: 26`, keeps only numeric fields, and gives up
    quietly (empty dict) if the report is not there.
    '''
    for line in text.splitlines():
        if "REG" not in line or ":" not in line:
            continue
        out = {}
        for k, v in re.findall(r"([A-Z][A-Z0-9_\[\]]*)\s*:\s*([0-9]+)", line):
            out[k] = int(v)
        if out:
            return out
    return {}


def census_sass(text, func=KERNEL):
    '''Static SASS instruction census for one kernel: (total, {opcode: count}).'''
    counts, total, inside = {}, 0, False
    for line in text.splitlines():
        if "Function :" in line:
            inside = func in line
            continue
        if not inside:
            continue
        m = re.match(r"\s*(?:[A-Za-z_.$][\w.$]*:\s*)?/\*[0-9a-f]{4,}\*/\s+"
                     r"(?:@!?U?PT?\d\s+)?([A-Z][A-Z0-9._]*)", line)
        if m:
            counts[m.group(1)] = counts.get(m.group(1), 0) + 1
            total += 1
    return total, counts


def parse_smi_trace(text):
    '''Rows out of `nvidia-smi --query-gpu=... --format=csv -lms N`.'''
    rows = []
    for line in text.splitlines():
        if not line.strip() or line.lower().startswith("timestamp"):
            continue
        parts = [p.strip() for p in line.split(",")]
        if len(parts) < 6:
            continue
        row = {"ts": parts[0], "ts_epoch": _smi_ts(parts[0]), "gpu_util": _num(parts[1]),
               "mem_util": _num(parts[2]), "sm_mhz": _num(parts[3]), "mem_mhz": _num(parts[4]),
               "power_w": _num(parts[5]),
               "temp_c": _num(parts[6]) if len(parts) > 6 else None}
        if row["gpu_util"] is None:
            continue
        rows.append(row)
    return rows


def occupancy(regs_per_thread, block, props):
    '''Theoretical occupancy for block threads per block, one thread per path.'''
    if not regs_per_thread or not props.get("sms"):
        return {}
    regs_per_block = ((regs_per_thread * 32 + 255) // 256) * 256 * (block // 32)
    limits = {
        "registers": props["regs_per_sm"] // regs_per_block if regs_per_block else 0,
        "threads": props["max_threads_per_sm"] // block,
        "block slots": 32,
    }
    resident = max(0, min(limits.values()))
    warps_per_sm = props["max_threads_per_sm"] // max(1, props.get("warp_size", 32))
    return {"regs_per_block": regs_per_block, "limits": limits,
            "limiter": min(limits, key=lambda k: limits[k]),
            "resident_blocks_per_sm": resident,
            "resident_warps_per_sm": resident * (block // 32),
            "warps_per_sm": warps_per_sm,
            "theoretical_occupancy_pct": 100.0 * resident * (block // 32) / warps_per_sm}


# ---- device properties (already dumped by ./build/gbm info in cell 3) -------
props = {}
env_file = RES / "environment_cuda.txt"
if env_file.exists():
    env_txt = env_file.read_text(encoding="utf-8", errors="replace")
    for key, pat in (("device", r"device\s*:\s*(.+)"),
                     ("cc", r"compute capability\s*:\s*(\d+\.\d+)"),
                     ("sms", r"multiprocessors \(SMs\)\s*:\s*(\d+)"),
                     ("max_threads_per_sm", r"max threads per SM\s*:\s*(\d+)"),
                     ("max_threads_per_block", r"max threads per block\s*:\s*(\d+)"),
                     ("smem_per_block", r"shared memory per block\s*:\s*(\d+)"),
                     ("warp_size", r"warp size\s*:\s*(\d+)"),
                     ("regs_per_sm", r"registers per SM\s*:\s*(\d+)"),
                     ("regs_per_thread", r"registers per thread\s*:\s*(\d+)"),
                     ("l2_bytes", r"L2 cache size\s*:\s*(\d+)"),
                     ("mem_clock_mhz", r"memory clock rate\s*:\s*([\d.]+)")):
        m = re.search(pat, env_txt)
        if not m:
            continue
        v = m.group(1).strip()
        props[key] = v if key == "device" else (float(v) if "." in v else int(v))

# ---- tool discovery --------------------------------------------------------
TOOLS = {}
for name in ("nsys", "ncu", "nvprof", "cuobjdump", "nvdisasm", "ptxas", "nvidia-smi"):
    TOOLS[name] = find_tool(name)

prof_lines = ["# Profiling evidence", "",
              "GPU: %s (%s)%s | nvcc: %s | runtime: %s"
              % (GPU_NAME, ARCH,
                 (", %d SMs, %d threads/SM, %d regs/SM" % (props["sms"],
                                                           props["max_threads_per_sm"],
                                                           props["regs_per_sm"]))
                 if props.get("sms") else "", NVCC, sys.version.split()[0]), "",
              "Representative workload: `./build/gbm rng --paths %d --block %d --steps %d "
              "--reps %d` (the real kernel, 252 Euler steps per path, terminal price only)."
              % (REP[0], REP[1], REP[2], REP[3]), "",
              "Every number below is labelled **measured** (a tool produced it here), **static** "
              "(read out of the built cubin) or **estimated** (derived, with its assumption "
              "stated). Nothing here is a CUPTI hardware-counter result unless Nsight actually "
              "ran, which is reported in section 2.", "",
              "## 1. Tool discovery in this runtime", "",
              "| tool | resolved path |", "|---|---|"]
for name in sorted(TOOLS):
    prof_lines.append("| `%s` | %s |" % (name, TOOLS[name] or "not found"))
prof_lines.append("")
print("\ntool discovery:", {k: bool(v) for k, v in TOOLS.items()})

# ---- 2. Nsight, if this image actually has it ------------------------------
raw_files = []

if TOOLS["nsys"]:
    cmd = [TOOLS["nsys"], "profile", "--force-overwrite=true", "--stats=true", "-o",
           "results/nsys_gbm", "./build/gbm", "rng", "--paths", str(REP[0]), "--block",
           str(REP[1]), "--steps", str(REP[2]), "--reps", "1", "--out", str(PROFILE_DIR)]
    out, rc = run_tool("nsys", cmd, timeout=900, cap=4000)
    pathlib.Path("results/nsys_stdout.txt").write_text(out)
    raw_files.append("results/nsys_stdout.txt")
    prof_lines += ["## 2. Nsight Systems (`nsys`) - present, exit code %d" % rc, "",
                   "```", " ".join(cmd), "```", "", "```", out[-4000:], "```", ""]
    prof_lines += (["- **Nsight Systems produced a usable timeline.**"] if rc == 0 else
                   ["- **Nsight Systems did not produce results here**; see the output above."])
    prof_lines.append("")
else:
    prof_lines += ["## 2. Nsight Systems (`nsys`) - not in this image", "",
                   "`nsys` is absent, so no CUPTI timeline could be captured here. The commands "
                   "that produce one on a machine that has it are in section 6.", ""]
    print("nsys not available in this environment")

if TOOLS["ncu"]:
    cmd = [TOOLS["ncu"], "--set", "basic", "--launch-count", "1", "--kernel-name",
           "regex:" + KERNEL, "./build/gbm", "rng", "--paths", str(REP[0]), "--block",
           str(REP[1]), "--steps", str(REP[2]), "--reps", "1", "--out", str(PROFILE_DIR)]
    out, rc = run_tool("ncu", cmd, timeout=900, cap=6000)
    pathlib.Path("results/ncu_stdout.txt").write_text(out)
    raw_files.append("results/ncu_stdout.txt")
    prof_lines += ["## 3. Nsight Compute (`ncu`) - present, exit code %d" % rc, "",
                   "```", " ".join(cmd), "```", ""]
    if "ERR_NVGPUCTRPERM" in out:
        prof_lines += ["**The container denies GPU performance counters** (verbatim):", "",
                       "```", "\n".join(l for l in out.splitlines()
                                      if "ERROR" in l or "NVGPUCTRPERM" in l), "```", "",
                       "So `ncu` cannot be used inside Kaggle: reading those counters needs "
                       "`CAP_SYS_ADMIN` / `NVreg_RestrictProfilingToAdminUsers=0`. Section 4 "
                       "replaces it with evidence that the kernel *can* provide here.", ""]
    else:
        prof_lines += ["```", out[-6000:], "```", ""]
else:
    prof_lines += ["## 3. Nsight Compute (`ncu`) - not in this image", "",
                   "`ncu` is absent or unusable here, so no hardware counters were read.", ""]

# ---- 4. What does work here -----------------------------------------------
prof_lines += ["## 4. Kaggle-compatible kernel profile (everything below ran here)", ""]
try:
    # 4a. measured: one representative kernel run, CUDA events, kernel-only timing
    rep_cmd = ["./build/gbm", "rng", "--paths", str(REP[0]), "--block", str(REP[1]),
               "--steps", str(REP[2]), "--reps", str(REP[3]), "--out", str(PROFILE_DIR)]
    rep_out, rep_rc = run_tool("representative kernel run", rep_cmd, timeout=900)
    rep = {}
    rep_csv = PROFILE_DIR / "rng_overhead.csv"
    if rep_csv.exists():
        rep = pd.read_csv(rep_csv).iloc[0].to_dict()
    prof_lines += ["### 4a. Representative run - **measured** (CUDA events, kernel only)", "",
                   "```", " ".join(rep_cmd), "```", ""]
    if rep:
        sec = float(rep.get("kernel_ms", 0.0)) / 1000.0
        exps = REP[0] * REP[2] / sec if sec > 0 else 0.0
        prof_lines += ["| kernel ms | us / step | ns / path / step | MPaths/s | expf calls | expf/s |",
                       "|---|---|---|---|---|---|",
                       "| %.4f | %.4f | %.4f | %.2f | %d | %.3e |"
                       % (float(rep.get("kernel_ms", 0.0)), float(rep.get("us_per_step", 0.0)),
                          float(rep.get("ns_per_path_step", 0.0)),
                          float(rep.get("gpu_paths_per_sec", 0.0)) / 1e6,
                          REP[0] * REP[2], exps), "",
                       "`expf` calls are exact (%d paths x %d steps - one `expf` per Euler step), "
                       "so the expf/s column is a directly measured throughput; the time is the "
                       "median of %d CUDA-event runs and excludes the host-side copy."
                       % (REP[0], REP[2], REP[3]), ""]
        print("representative run: kernel %.4f ms, %.3e expf/s" % (float(rep.get("kernel_ms", 0.0)), exps))
    else:
        prof_lines += ["**The representative run wrote no CSV** (exit code %d); its console output "
                       "is above." % rep_rc, ""]

    # 4b. static: kernel resources, from the cubin that cell 3 produced
    res_usage, ptxas = "", {}
    if TOOLS["cuobjdump"]:
        res_usage, _ = run_tool("cuobjdump -res-usage",
                               [TOOLS["cuobjdump"], "-res-usage", "build/gbm"], timeout=300)
        pathlib.Path("results/cuobjdump_res_usage.txt").write_text(res_usage)
        raw_files.append("results/cuobjdump_res_usage.txt")
    compile_log = RES / "compile_log.txt"
    if compile_log.exists():
        ptxas = parse_ptxas_v(compile_log.read_text(encoding="utf-8", errors="replace"))
    usage = parse_res_usage(res_usage) or ptxas
    regs = int(usage.get("REG", usage.get("registers_per_thread", props.get("regs_per_thread", 0))) or 0)
    prof_lines += ["### 4b. Kernel resource usage - **static** (from the built cubin)", ""]
    if usage:
        prof_lines += ["| registers / thread | static smem | stack | spill stores | spill loads | source |",
                       "|---|---|---|---|---|---|",
                       "| %s | %s | %s | %s | %s | %s |"
                       % (usage.get("REG", ptxas.get("registers_per_thread", "?")),
                          usage.get("SHARED", ptxas.get("static_smem_bytes", "?")),
                          usage.get("STACK", ptxas.get("stack_bytes", "?")),
                          ptxas.get("spill_stores_bytes", 0) if ptxas else "?",
                          ptxas.get("spill_loads_bytes", 0) if ptxas else "?",
                          "cuobjdump -res-usage" if parse_res_usage(res_usage) else "nvcc -Xptxas -v"),
                       "",
                       "No spill and no local memory means the inner loop lives entirely in "
                       "registers, which is what a one-thread-per-path stencil like this needs.", ""]
    else:
        prof_lines += ["Neither `cuobjdump -res-usage` nor `-Xptxas -v` produced register counts "
                       "here.", ""]

    # 4c. static: SASS instruction census
    census = {}
    if TOOLS["cuobjdump"]:
        sass, _ = run_tool("cuobjdump -sass", [TOOLS["cuobjdump"], "-sass", "build/gbm"],
                           timeout=600, cap=400)
        total, census = census_sass(sass)
        if census:
            with open("results/sass_census.csv", "w") as fh:
                fh.write("opcode,count\n")
                for op, c in sorted(census.items(), key=lambda kv: -kv[1]):
                    fh.write("%s,%d\n" % (op, c))
            raw_files.append("results/sass_census.csv")
            prof_lines += ["### 4c. SASS instruction census - **static** (`cuobjdump -sass`)", "",
                           "%d SASS instructions in `%s`." % (total, KERNEL), "",
                           "| opcode | count | share |", "|---|---|---|"]
            for op, c in sorted(census.items(), key=lambda kv: -kv[1])[:12]:
                prof_lines.append("| `%s` | %d | %.1f%% |" % (op, c, 100.0 * c / total))
            prof_lines += ["",
                           "These are instruction counts in the *compiled kernel body as written*, "
                           "not executed counts: `#pragma unroll 4` (and the `curand_normal2` "
                           "pairing) means the loop body in the SASS is several Euler steps wide "
                           "and executes many times per path. Nothing on this line divides by the "
                           "logical 252 steps, because that ratio would be an artefact of the "
                           "unroll factor rather than a measurement.", ""]
        else:
            prof_lines += ["`cuobjdump -sass` ran but no SASS was found for `%s`." % KERNEL, ""]

    # 4d. derived: occupancy / launch geometry
    occ = occupancy(regs, REP[1], props)
    if occ:
        grid_blocks = (REP[0] + REP[1] - 1) // REP[1]
        waves = grid_blocks / float(props["sms"] * max(1, occ["resident_blocks_per_sm"]))
        prof_lines += ["### 4d. Occupancy and launch geometry - **derived** from 4b/4c", "",
                       "| thread / path | regs / thread | block | regs / block | resident blocks / SM | "
                       "resident warps / SM | warps / SM | theoretical occupancy | limiter |",
                       "|---|---|---|---|---|---|---|---|---|",
                       "| 1 | %d | %d | %d | %d | %d | %d | %.1f%% | %s |"
                       % (regs, REP[1], occ["regs_per_block"], occ["resident_blocks_per_sm"],
                          occ["resident_warps_per_sm"], occ["warps_per_sm"],
                          occ["theoretical_occupancy_pct"], occ["limiter"]), "",
                       "| launch (representative run) | blocks | waves | blocks / SM |",
                       "|---|---|---|---|",
                       "| %d paths, block %d | %d | %.1f | %.1f |"
                       % (REP[0], REP[1], grid_blocks, waves,
                          grid_blocks / float(props["sms"])), "",
                       "Register allocation is rounded to the 256-register warp granularity. "
                       "Occupancy is the *theoretical* figure the hardware allows for this "
                       "kernel and block size - without `ncu` it cannot be compared against the "
                       "achieved occupancy, and that gap is stated in section 5.", ""]
    else:
        prof_lines += ["### 4d. Occupancy - not computable here", "",
                       "Needs a register count (4b) and device properties from "
                       "`results/environment_cuda.txt`; one of the two was missing.", ""]

    # 4e. measured: sampled nvidia-smi telemetry during a soak run
    if TOOLS["nvidia-smi"]:
        q = ("timestamp,utilization.gpu,utilization.memory,clocks.sm,clocks.mem,"
             "power.draw,temperature.gpu")
        smi = subprocess.Popen([TOOLS["nvidia-smi"], "--query-gpu=" + q, "--format=csv",
                                "-lms", "50"], stdout=subprocess.PIPE,
                               stderr=subprocess.DEVNULL, text=True)
        lines = []
        # Read the sampler's pipe on a thread: nvidia-smi's own buffering would
        # otherwise lose the tail of the trace when the sampler is stopped.
        reader = threading.Thread(target=lambda: [lines.append(l) for l in smi.stdout],
                                  daemon=True)
        reader.start()
        marker, busy_end = None, None
        try:
            time.sleep(1.0)  # pre-soak window (not guaranteed idle on a shared GPU)
            marker = time.time()
            soak = ["./build/gbm", "rng", "--paths", str(REP[0]), "--block", str(REP[1]),
                    "--steps", str(REP[2]), "--reps", "150", "--out", str(PROFILE_DIR)]
            run_tool("soak run for the telemetry trace", soak, timeout=900)
            busy_end = time.time()
            time.sleep(0.5)
        finally:
            smi.terminate()
            try:
                smi.wait(timeout=30)
            except subprocess.TimeoutExpired:
                smi.kill()
            reader.join(timeout=10)
        pathlib.Path("results/nvidia_smi_trace.csv").write_text("".join(lines))
        raw_files.append("results/nvidia_smi_trace.csv")
        trace = parse_smi_trace("".join(lines))
        if marker is not None and busy_end is not None:
            before = [r for r in trace if r["ts_epoch"] is None or r["ts_epoch"] < marker]
            soak = [r for r in trace if r["ts_epoch"] is not None
                    and marker <= r["ts_epoch"] <= busy_end]
        else:
            before, soak = [], trace
        if not soak:  # unparsable or missing timestamps: split on utilisation instead
            soak = [r for r in trace if (r["gpu_util"] or 0) >= 20]
            before = [r for r in trace if (r["gpu_util"] or 0) < 20]

        def _stat(rows, key, how):
            vals = [r[key] for r in rows if r.get(key) is not None]
            if not vals:
                return "n/a"
            a = np.array(vals, dtype=float)
            return "%.0f" % {"mean": a.mean(), "median": float(np.median(a)),
                             "peak": a.max()}[how]

        def _row(label, rows):
            return ("| %s | %d | %s%% | %s%% | %s%% | %s%% | %s MHz | %s W | %s W | %s C |"
                    % (label, len(rows), _stat(rows, "gpu_util", "mean"),
                       _stat(rows, "gpu_util", "median"), _stat(rows, "gpu_util", "peak"),
                       _stat(rows, "mem_util", "peak"), _stat(rows, "sm_mhz", "peak"),
                       _stat(rows, "power_w", "mean"), _stat(rows, "power_w", "peak"),
                       _stat(rows, "temp_c", "peak")))

        prof_lines += ["### 4e. Sampled GPU telemetry around the kernel - **measured**", "",
                       "`nvidia-smi --query-gpu=... --format=csv -lms 50` sampled across two "
                       "windows: the second before a soak run (`--reps 150`) of the same kernel "
                       "geometry starts, and the soak itself. Two caveats stated up front: this "
                       "is a *sampled* trace of the GPU's own counters, not a CUPTI timeline, so "
                       "it cannot show individual launches; and on a shared Kaggle GPU "
                       "`utilization.gpu` also counts what other tenants are doing, so the "
                       "'before' window is *not* guaranteed idle and its peak is not a baseline "
                       "of our own. The informative signal is the change in the distribution "
                       "(and in power draw), not either window's peak on its own.", "",
                       "| window | samples | SM util mean | SM util median | SM util peak | "
                       "mem util peak | SM clock peak | power mean | power peak | temp peak |",
                       "|---|---|---|---|---|---|---|---|---|---|",
                       _row("before the soak (1 s)", before), _row("soak window", soak),
                       "", "Raw samples: `results/nvidia_smi_trace.csv`.", ""]
        print("telemetry: %d samples before / %d in the soak window" % (len(before), len(soak)))

    # 4f. cross-checks: does the static picture match what the run actually did?
    if census:
        mufu = sum(c for op, c in census.items() if op.startswith("MUFU"))
        local = sum(c for op, c in census.items() if op in ("LDL", "STL", "LDS", "STS"))
        ldg = sum(c for op, c in census.items() if op.startswith("LDG"))
        stg = sum(c for op, c in census.items() if op.startswith("STG"))
        expf_calls = REP[0] * REP[2]
        prof_lines += ["### 4f. Static-vs-measured cross-checks - **checked against this run**", "",
                       "| check | static observation | verdict |", "|---|---|---|",
                       "| `expf` is a hardware instruction | %d `MUFU` instruction(s) in the kernel "
                       "body, while the run performs exactly %d `expf` calls (%d paths x %d steps) "
                       "| %s |" % (mufu, expf_calls, REP[0], REP[2],
                                   "consistent" if mufu > 0 else "no MUFU: expf is emulated"),
                       "| the inner loop is register-resident | %d local/shared-memory "
                       "instruction(s) in the body | %s |"
                       % (local, "no local-memory traffic, so the loop does not spill" if local == 0
                          else "local memory traffic present"),
                       "| global traffic is one terminal store per path | %d `LDG`, %d `STG` "
                       "instruction(s) in the body | %s |"
                       % (ldg, stg, "paths live in registers and only the terminal price is "
                          "written, as designed" if stg > 0 and ldg <= 1
                          else "unexpected memory pattern"),
                       "| measured throughput the census has to support | %s | from 4a, same run |"
                       % (("%.3e paths/s" % float(rep.get("gpu_paths_per_sec", 0.0)))
                          if rep else "run produced no CSV"), "",
                       "These cross-checks are what the static data can honestly support. "
                       "Converting the census into GFLOP/s or a share of peak would require the "
                       "*executed* per-step instruction count of the unrolled loop - precisely "
                       "the data `ncu` provides and this container blocks - so it is not claimed "
                       "here.", ""]
    print("\nstatic/derived analysis done")
except Exception as exc:  # the profiling ladder must never sink the deliverables
    import traceback
    tb = traceback.format_exc()
    prof_lines += ["**The Kaggle-compatible analysis failed** - recorded verbatim so the rest of "
                   "the run is unaffected:", "", "```", tb[-3000:], "```", ""]
    print("profiling analysis failed:\n" + tb)

# ---- 5/6. limitations + how to get the real thing ---------------------------
prof_lines += ["## 5. Limitations of this profiling (stated, not hidden)", "",
               "- **No hardware-counter data.** Reading SM/L2/DRAM counters needs "
               "`CAP_SYS_ADMIN` / `NVreg_RestrictProfilingToAdminUsers=0`; inside Kaggle `ncu` "
               "returns `ERR_NVGPUCTRPERM` and `nsys` is not installed. Nothing in this file is "
               "a CUPTI hardware-counter measurement, and no section above is labelled as one.",
               "- **What replaces it:** CUDA-event kernel timing (measured), static resource and "
               "SASS statistics from the built cubin (static), occupancy derived from those two "
               "(derived), and sampled `nvidia-smi` telemetry at 50 ms (measured but coarse - it "
               "shows the device busy during the kernel phase, not individual launches).",
               "- **Achieved occupancy, cache hit rates, DRAM traffic, stall reasons and IPC are "
               "not available** without Nsight Compute. Theoretical occupancy in 4d is a hardware "
               "limit, not a measurement.",
               "- The instruction mix in 4c/4f is *static*: it counts SASS instructions, not "
               "executed instructions, so a divergent or partially predicated path would not show "
               "up in it. This kernel has no data-dependent branches, so the two coincide here.",
               "- CUDA-event timings (`benchmark_*.csv`, 4a) remain the primary timing evidence and "
               "are never presented as Nsight results.", "", "Captured raw artifacts:", ""] + \
              ["- `%s`" % f for f in raw_files] + ["- `results/profile/rng_overhead.csv` (the "
               "representative run's own output, kept out of `results/rng_overhead.csv` so the "
               "step sweep from cell 3 stays intact)", ""]

prof_lines += ["## 6. Getting the full picture on a permitted machine", "",
               "```bash", "make ARCH=sm_86",
               "# timeline: kernel launches, gaps, host/device overlap",
               "nsys profile --stats=true -o nsys_gbm ./build/gbm bench --sizes 1000000 \\",
               "    --blocks 256 --steps 252 --reps 5 --out results",
               "# counters: occupancy achieved, stalls, DRAM/L2 traffic, IPC",
               "ncu --set full --kernel-name regex:%s -o ncu_gbm \\" % KERNEL,
               "    ./build/gbm rng --paths 1000000 --block 256 --steps 252 --reps 1 --out results",
               "```", ""]

(RES / "profiling.md").write_text("\n".join(prof_lines))
print("\nwrote results/profiling.md (%d bytes)" % (RES / "profiling.md").stat().st_size)

# ---- machine-generated summary -------------------------------------------
import re

fit_txt = (RES / "rng_overhead_fit.txt").read_text().strip() \
    if (RES / "rng_overhead_fit.txt").exists() else "(no fit written)"


def md_table(df, cols, headers, fmts):
    out = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
    for _, row in df.iterrows():
        out.append("| " + " | ".join(f(row[c]) for c, f in zip(cols, fmts)) + " |")
    return "\n".join(out)


size_tbl = md_table(
    sizes_g,
    ["paths", "steps", "block", "cpu_ms", "gpu_kernel_ms", "gpu_e2e_ms",
     "speedup_kernel", "speedup_e2e", "gpu_paths_per_sec"],
    ["paths", "steps", "block", "CPU ms", "GPU kernel ms", "GPU e2e ms",
     "speedup (kernel)", "speedup (e2e)", "GPU paths/s"],
    [lambda v: format(int(v), ","), lambda v: "%d" % v, lambda v: "%d" % v,
     lambda v: "%.2f" % v, lambda v: "%.3f" % v, lambda v: "%.3f" % v,
     lambda v: "%.1fx" % v, lambda v: "%.1fx" % v, lambda v: "%.3e" % v])

block_tbl = md_table(
    blocks_g.sort_values(["paths", "block"]),
    ["paths", "block", "gpu_kernel_ms", "gpu_e2e_ms", "gpu_paths_per_sec", "reps"],
    ["paths", "block", "GPU kernel ms", "GPU e2e ms", "GPU paths/s", "reps"],
    [lambda v: format(int(v), ","), lambda v: "%d" % v, lambda v: "%.3f" % v,
     lambda v: "%.3f" % v, lambda v: "%.3e" % v, lambda v: "%d" % v])

header = ("GPU: %s (compute capability %s) | CPU: %s (%s logical cores) | nvcc: %s\n\n"
          "Simulation: S0=100, mu=0.05, sigma=0.20, T=1 year, 252 steps, terminal price only;\n"
          "timings are medians (CPU: 3 reps, GPU: 5 reps) from CUDA events on a single Kaggle GPU."
          % (GPU_NAME, ARCH.replace("sm_", ""), cpu_model, n_cpu, NVCC))

md_block = "\n".join([
    header, "",
    "### CPU vs GPU over simulation size (block size 256)", "", size_tbl, "",
    "### GPU kernel time and throughput vs CUDA block size", "", block_tbl, "",
    "### RNG setup / launch-overhead fit", "", "```", fit_txt, "```", "",
    "Raw data: `results/benchmark_sizes.csv`, `results/benchmark_blocks.csv`, "
    "`results/rng_overhead.csv`. Plots: `results/plot_cpu_vs_gpu_time.png`, "
    "`results/plot_block_sizes.png`, `results/plot_speedup.png`.", "",
])

summary = "\n".join([
    "# Measured results (generated by the notebook - do not edit by hand)", "",
    md_block, "## Files", "", "```",
    "\n".join("%-34s %9d bytes" % (f.name, f.stat().st_size) for f in sorted(RES.glob("*"))
               if f.is_file()),
    "```", "",
])
(RES / "summary.md").write_text(summary)

# Patch the README between the RESULTS markers so the repo carries real numbers.
readme = pathlib.Path("README.md")
if readme.exists():
    txt = readme.read_text(encoding="utf-8")
    if "<!-- RESULTS:START -->" in txt and "<!-- RESULTS:END -->" in txt:
        txt = re.sub(r"<!-- RESULTS:START -->.*?<!-- RESULTS:END -->",
                     "<!-- RESULTS:START -->\n" + md_block + "<!-- RESULTS:END -->",
                     txt, flags=re.S)
        readme.write_text(txt, encoding="utf-8")
        print("patched README.md between the RESULTS markers")
    else:
        print("README.md has no RESULTS markers; paste results/summary.md manually")
else:
    print("no README.md in the working directory; paste results/summary.md manually")

print("\n" + "=" * 74)
print("MEASURED RESULTS (also written to results/summary.md)")
print("=" * 74)
print(summary)

In [ ]:
# ============ 5. Package the deliverables and download them ================
# Runs last, and only if the pipeline really produced what this project promises:
# every required artifact exists and is not truncated, the correctness CSV parses
# with the expected schema, and no check FAILed. Otherwise it prints exactly what
# is missing or wrong and refuses to hand over a half-finished result set.
import os, pathlib, zipfile

from IPython.display import FileLink, display

RES = pathlib.Path("results").resolve()
WORK = pathlib.Path.cwd().resolve()

banner("gate: verify every deliverable before downloading anything")

# artifact -> smallest plausible size, which catches empty/truncated files
REQUIRED = {
    "environment.txt": 200,
    "environment_cuda.txt": 200,
    "compile_log.txt": 200,
    "smoke_stdout.txt": 20,
    "verify_stdout.txt": 200,
    "bench_stdout.txt": 200,
    "rng_stdout.txt": 200,
    "correctness.csv": 200,
    "benchmark_sizes.csv": 100,
    "benchmark_blocks.csv": 300,
    "rng_overhead.csv": 40,
    "rng_overhead_fit.txt": 200,
    "profiling.md": 200,
    "plot_cpu_vs_gpu_time.png": 4000,
    "plot_block_sizes.png": 4000,
    "plot_speedup.png": 2000,
}

missing, problems = [], []
for name, min_bytes in REQUIRED.items():
    f = RES / name
    if not f.exists():
        missing.append(name)
        continue
    n = f.stat().st_size
    if n < min_bytes:
        problems.append("%s is %d bytes, expected >= %d - looks truncated" % (name, n, min_bytes))
    else:
        print("  ok  %-26s %9d bytes" % (name, n))
if missing:
    print("  MISSING: %s" % ", ".join(missing))

# Freshness: everything above must have been written by THIS session, not left
# behind by an earlier (possibly failed) run that cell 1..4 then failed to redo.
run_start = globals().get("RUN_START")
if run_start is None:
    print("  note  RUN_START is not set (cell 1 did not run in this session) - skipping the "
          "freshness check")
else:
    stale = [n for n in REQUIRED
             if (RES / n).exists() and (RES / n).stat().st_mtime < run_start - 5.0]
    if stale:
        problems.append("left over from an earlier session, not rewritten by this run: %s"
                        % ", ".join(stale))
    else:
        print("  ok  %-26s written after cell 1 started" % "all artifacts are fresh")

# summary.md is written last in cell 4, so it must also list the measured tables.
summary_md = RES / "summary.md"
if summary_md.exists():
    text = summary_md.read_text(encoding="utf-8", errors="replace")
    if "### CPU vs GPU over simulation size" not in text or "### RNG setup" not in text:
        problems.append("summary.md is missing one of the measured tables")
    else:
        print("  ok  %-26s %9d bytes, tables present" % ("summary.md", summary_md.stat().st_size))

# The correctness CSV must be strict CSV with no FAIL row, and it must contain GPU
# results: this notebook only makes sense on a GPU session.
CORRECTNESS_COLUMNS = ["check", "backend", "paths", "steps", "seed", "value", "expected",
                       "rel_err", "z_score", "status", "note"]
csv_path = RES / "correctness.csv"
if csv_path.exists():
    try:
        import pandas as pd
        checks = pd.read_csv(csv_path)
    except Exception as exc:
        problems.append("correctness.csv does not parse as CSV: %s" % exc)
    else:
        if list(checks.columns) != CORRECTNESS_COLUMNS:
            problems.append("correctness.csv columns are %s, expected %s"
                            % (list(checks.columns), CORRECTNESS_COLUMNS))
        elif checks.empty:
            problems.append("correctness.csv has no rows")
        else:
            status = checks["status"].astype(str).str.upper()
            backend = checks["backend"].astype(str).str.lower()
            failed = checks[status == "FAIL"]
            n_gpu_pass = int(((backend == "gpu") & (status == "PASS")).sum())
            if len(failed):
                problems.append("%d correctness row(s) FAILED: %s"
                                % (len(failed), ", ".join(sorted(set(failed["check"].astype(str))))))
            if n_gpu_pass == 0:
                problems.append("no PASS row with backend=gpu - this notebook is meant to run on "
                                "a GPU session")
            print("  ok  %-26s %d rows: %d PASS, %d SKIP, %d FAIL, %d GPU PASS"
                  % ("correctness.csv", len(checks), int((status == "PASS").sum()),
                     int((status == "SKIP").sum()), len(failed), n_gpu_pass))

if missing or problems:
    detail = ["incomplete: results/%s was never written" % m for m in missing]
    detail += ["problem: %s" % p for p in problems]
    print("\n" + "=" * 74)
    print("NOT DOWNLOADING - the run did not produce a complete, passing result set")
    print("=" * 74)
    for d in detail:
        print("  " + d)
    raise RuntimeError("result set is incomplete or wrong (%d missing, %d problem(s)): %s"
                       % (len(missing), len(problems), "; ".join(detail)))

# ---- package everything into one archive -----------------------------------
zip_path = WORK / "gbm_montecarlo_results.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(str(zip_path), "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RES.rglob("*")):
        if f.is_file():
            z.write(str(f), str(f.relative_to(WORK)))
    source = WORK / "src" / "gbm_montecarlo.cu"
    if source.exists():  # keep the exact source that produced these numbers
        z.write(str(source), str(source.relative_to(WORK)))
with zipfile.ZipFile(str(zip_path)) as z:
    entries = sorted(z.infolist(), key=lambda i: i.filename)
    packed = sum(i.file_size for i in entries)
print("\npackaged %d files (%.1f KiB raw) -> %s (%.1f KiB)"
      % (len(entries), packed / 1024.0, zip_path.name, zip_path.stat().st_size / 1024.0))

# ---- hand the files over ---------------------------------------------------
# One best-effort browser download for the archive, then a FileLink for every
# deliverable. Kaggle's client does not define the Colab `download` JS helper: it
# logs "Javascript Error: download is not defined" while (observed on a T4 session)
# still fetching the file, and that outcome is invisible from Python. So this never
# claims a download succeeded, and the links are always there as the reliable path.
IS_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_GPU"))
IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE")
                 or pathlib.Path("/kaggle/working").exists())
print("runtime   : %s" % ("Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE
                          else "neither Colab nor Kaggle detected"))


def js_download(path):
    '''Ask the notebook client to download a file. True if the helper was called.'''
    try:
        from google.colab import files as colab_files
    except ImportError:
        return False
    try:
        colab_files.download(str(path))
    except Exception as exc:
        print("  note  JS download attempt raised: %s" % exc)
        return False
    return True


def deliver(path, label, try_js=False):
    p = pathlib.Path(path).resolve()
    if try_js and js_download(p):
        print("  asked the notebook client for %s - if no download arrived, use the link below%s"
              % (p.name, " (Kaggle logs a harmless 'download is not defined' JS error)"
                 if IS_KAGGLE else ""))
    try:
        href = str(p.relative_to(WORK))
    except ValueError:
        href = str(p)
    display(FileLink(href, result_html_prefix="download %s -> " % label))


print("\n" + "=" * 74)
print("ALL GREEN - handing over the results")
print("=" * 74)
deliver(zip_path, "everything (zip)", try_js=True)
for name in ("summary.md", "correctness.csv", "benchmark_sizes.csv", "benchmark_blocks.csv",
             "rng_overhead.csv", "profiling.md", "environment.txt", "plot_cpu_vs_gpu_time.png",
             "plot_block_sizes.png", "plot_speedup.png"):
    f = RES / name
    if f.exists():
        deliver(f, name)

print("\nNo-clicking alternative: Save Version -> Save & Run All, then the notebook's")
print("Output tab contains %s and the whole results/ directory." % zip_path.name)
print("\narchive contents:")
for info in entries:
    print("  %-34s %9d bytes" % (info.filename, info.file_size))